# Gemma 270M Colab V3.6 — Muon

Select **TPU v5e-1** in Colab, then **Run all**. One fixed **60-second** attempt.
The notebook is self-contained; no source upload or configuration editing is
needed for the default Gemma tokenizer. Hugging Face access to that repository
is required. HF_TOKEN can be supplied through Colab Secrets or the password prompt.

**All-parameter Muon:** every parameter, including the tied embedding/output
table and normalization vectors, goes through the public low-level Muon transform.
There is **no AdamW branch**. Vectors/scalars are reshaped to **1 x N**, transformed,
and restored to their original shape. This is an explicitly experimental extension
of matrix Muon. Learning rate **0.0015**, momentum **0.95**, Nesterov, **5**
Newton-Schulz steps and consistent-RMS factor **0.2**. The nominal learning-rate
times RMS factor is **0.0003**; actual updates need not have exactly that RMS.
Weight decay **0.01** applies to matrix/tensor weights, not normalization vectors.
The **167,774,720-parameter** tied table is included for the default tokenizer,
and its orthogonalization can increase step and compilation time substantially.
This notebook is not represented as a known speed improvement.

Muon implementation reference: [pinned Optax 0.2.8 source](https://github.com/google-deepmind/optax/blob/v0.2.8/optax/contrib/_muon.py).


Both new versions use native Gemma attention and dense exact softmax, **batch 2 x
1024**, seed **42**, the same V3.4 frozen raw/token caches, CPU training-byte
accounting, one combined held-out evaluator and one FP32 weights-only artifact.
Splash and Tokamax do not execute. The existing dependency environment is retained
to avoid another package-resolution experiment. New Drive outputs use
`ShiningGem/colab-v5e1/V3.6/<unique run id>`.

The FP32 weights are approximately **1.07 GB** for the default tokenizer.
**Optimizer state is deliberately not saved; exact optimizer resumption is not
supported.** Reports and tokenizer files are retained. Connected setup,
compilation, evaluation and saving consume additional CU outside the 60 seconds.
One notebook does not automatically run the other. **No local checks or tests
were run. TPU results are pending your execution.**


# Gemma Colab V3.6 engineering requirements

Fixed configuration: one v5e-1 TPU, seed 42, batch 2 x 1024, 60 seconds,
at most 16 updates per dispatch, one attempt, random initialization.

| ID | Requirement | Verification |
|---|---|---|
| V36-001 | The runner shall make one attempt with no automatic retry or search. | Exclusive training.started marker; preserved prior logs. |
| V36-002 | The runner shall use exactly one v5e TPU. | Device platform, kind and count at train startup. |
| V36-003 | All source revisions shall be pinned. | Frozen shared source lock and source set. |
| V36-004 | The raw corpus shall remain the V3 static-byte mixture. | Protocol and raw/token manifest SHA256 checks. |
| V36-005 | Training and evaluation shall use disjoint source groups and exact spans. | Corpus builder deduplication and split separation; not an exhaustive near-duplicate guarantee. |
| V36-006 | Tokenization shall reconstruct every selected original UTF-8 byte. | Round-trip and byte-count equality; no corpus modification. |
| V36-007 | Packing shall preserve targets, chunk context and reset positions. | Existing intact-packing checks and cached input hashes. |
| V36-008 | Native attention shall enforce causality, document isolation and safe padding queries. | Small mask comparison in Colab before training; no custom kernels. |
| V36-009 | Training inputs shall reside on the TPU before timing. | Device ownership and synchronization barrier. |
| V36-010 | The model shall retain the native Gemma backbone, tied table, FP32 masters and fewer than 300M parameters. | Parameter shapes/count/dtypes and shared backbone hash. |
| V36-011 | Every parameter shall use Muon; no parameter shall use the other optimizer. | Complete per-leaf optimizer routing and parameter totals. |
| V36-012 | No training dispatch shall start at or after 60 elapsed seconds. | Monotonic dispatch timestamps; actual overrun reported. |
| V36-013 | Each dispatch shall perform between 1 and 16 updates. | Requested and completed update counts. |
| V36-014 | Nonfinite loss or gradient shall stop training; final weights shall be finite. | Finite guard, failure evidence and final weight reduction. |
| V36-015 | Zero optimizer updates shall occur outside the timed interval. | Zero-update JIT preparation and optimizer/cursor counters. |
| V36-016 | Evaluation shall score the same bytes before and after training, including all 12 FineWiki languages. | Fixed inputs, original-byte denominators and separate EOS totals. |
| V36-017 | One FP32 weights artifact shall be persisted and checksum verified. | Local/Drive SHA256 equality, params-only schema and leaf count. |
| V36-018 | Training shall change a hidden transformer weight. | Initial/final weight probe. |
| V36-019 | At least 5 timed updates shall complete. | Final update count. |
| V36-020 | Processed bytes and tokens shall reconcile with completed batches. | CPU prefix sums, device token count, source/language equality. |
| V36-021 | Source preparation shall use no more than 6 workers. | Executor bound and recorded peak concurrency. |
| V36-022 | Completed source prefixes and cursors shall persist independently of final corpus completion. | Source cache state and checksums; warm raw corpus needs no fetch. |
| V36-023 | Worker completion order shall not determine corpus selection. | Canonical merge order with Wiki reserved first. |
| V36-024 | Cold preparation shall finish all 30 source jobs. | Source completion report, or verified complete raw cache. |
| V36-025 | The training batch shall be 2 sequences of 1024 tokens. | Resolved configuration; no batch search or fallback. |
| V36-026 | One combined evaluator shall report both held-out datasets separately. | One compiled evaluation function and runtime split identity. |
| V36-027 | Per-token source/language histograms shall not execute in timed training. | CPU prefix sums; training device input excludes accounting arrays. |
| V36-028 | The artifact shall declare optimizer resumption unsupported. | Weights-only schema and explicit report flag. |

The notebook shall reuse the exact successful V3.4 dependency lock; it shall not
resolve package versions again. A matching installed environment shall be reused.
Tokamax may remain installed in that lock but shall not execute in training.
Runtime startup shall perform hardware verification once, in the train process.

The notebook shall save reports and weights in a unique V3.6 run directory.
It shall preserve previous notebooks and prior attempt evidence. The artifact
builder shall use exclusive file creation. No local checks/tests shall be run.

Performance targets, not guarantees: at least 45,559.448 useful tokens/s,
at most 60 seconds compilation, at most 300 seconds notebook time with existing
data caches, at least 95% useful occupancy, and FineWiki macro BPB no higher than
2.782397528489431 for the exact V3.4 tokenizer/corpus. Performance-goal status,
engineering status and learning status shall be reported separately. These
targets also appear in V3.6; no claim is made that all-parameter Muon meets them.

The 60-second timer excludes preparation, compilation, evaluation and saving;
all of those consume paid connected runtime. At 2.92 CU/hour, exactly 60 seconds
is 0.0486667 CU and 300 seconds is 0.2433333 CU. No dollar conversion is assumed.
No successful TPU execution or restore is claimed by artifact construction.


## 1. Connect Drive and reuse the pinned environment
No model update or TPU initialization occurs here. A new run folder is created.


In [ ]:
REQUIREMENTS_INPUT = '\n# This file was autogenerated by uv via the following command:\n#    uv pip compile /tmp/gemma-colab-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_35 -o /tmp/gemma-colab-requirements.lock\nabsl-py==2.5.1\n    # via\n    #   array-record\n    #   chex\n    #   clu\n    #   dm-tree\n    #   etils\n    #   gemma\n    #   grain\n    #   kauldron\n    #   keras\n    #   ml-collections\n    #   optax\n    #   orbax-checkpoint\n    #   tensorboard\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorflow-metadata\n    #   xmanager\naiofile==3.12.3\n    # via py-key-value-aio\naiofiles==25.1.0\n    # via orbax-checkpoint\naiohappyeyeballs==2.7.1\n    # via aiohttp\naiohttp==3.14.5\n    # via\n    #   aiohttp-retry\n    #   cloud-sql-python-connector\n    #   fsspec\n    #   kubernetes\naiohttp-retry==2.9.1\n    # via kubernetes\naiosignal==1.4.0\n    # via aiohttp\nalembic==1.20.0\n    # via xmanager\naltair==6.3.0\n    # via kauldron\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.15.1\n    # via\n    #   google-genai\n    #   httpx\n    #   httpx2\n    #   mcp\n    #   py-key-value-aio\n    #   sse-starlette\n    #   starlette\n    #   watchfiles\nanywidget==0.11.0\n    # via dialog\narray-record==0.8.6\n    # via\n    #   grain\n    #   tensorflow-datasets\nasttokens==3.0.2\n    # via stack-data\nastunparse==1.6.3\n    # via tensorflow\nasync-generator==1.10\n    # via xmanager\nattrs==26.1.0\n    # via\n    #   aiohttp\n    #   cyclopts\n    #   dm-tree\n    #   jsonschema\n    #   jsonschema-path\n    #   referencing\n    #   xmanager\nauthlib==1.9.1\n    # via fastmcp-slim\nbeartype==0.23.1\n    # via py-key-value-aio\ncachetools==7.2.4\n    # via py-key-value-aio\ncaio==0.12.9\n    # via aiofile\ncertifi==2026.10.19\n    # via\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   httpcore\n    #   httpx\n    #   kubernetes\n    #   requests\ncffi==2.1.1\n    # via cryptography\ncharset-normalizer==3.5.2\n    # via requests\nchex==0.1.92\n    # via kauldron\nclick==8.5.0\n    # via\n    #   huggingface-hub\n    #   uvicorn\ncloud-sql-python-connector==1.23.0\n    # via xmanager\ncloudpickle==3.1.2\n    # via\n    #   grain\n    #   joblib\nclu==0.0.12\n    # via kauldron\ncomm==0.2.3\n    # via ipywidgets\ncontourpy==1.4.0\n    # via matplotlib\ncryptography==50.0.2\n    # via\n    #   authlib\n    #   cloud-sql-python-connector\n    #   google-auth\n    #   joserfc\n    #   pyjwt\n    #   secretstorage\ncycler==0.12.1\n    # via matplotlib\ncyclopts==5.2.0\n    # via fastmcp-slim\ndatasets==5.1.0\n    # via -r /tmp/gemma-colab-requirements.in\ndialog==1.1.0\n    # via gemma\ndill==0.4.1\n    # via\n    #   datasets\n    #   multiprocess\ndistro==1.9.0\n    # via google-genai\ndm-tree==0.1.10\n    # via tensorflow-datasets\ndnspython==2.9.0\n    # via\n    #   cloud-sql-python-connector\n    #   email-validator\ndocker==7.2.0\n    # via xmanager\ndocstring-parser==0.18.0\n    # via\n    #   cyclopts\n    #   google-cloud-aiplatform\n    #   simple-parsing\ndurationpy==0.11\n    # via kubernetes\neinops==0.8.2\n    # via\n    #   etils\n    #   gemma\n    #   kauldron\nemail-validator==2.3.0\n    # via pydantic\netils==1.14.0\n    # via\n    #   array-record\n    #   clu\n    #   dialog\n    #   gemma\n    #   grain\n    #   kauldron\n    #   orbax-checkpoint\n    #   tensorflow-datasets\n    #   xmanager\nexceptiongroup==1.3.1\n    # via fastmcp-slim\nexecuting==2.3.0\n    # via stack-data\nfastmcp==4.1.3\n    # via gemma\nfastmcp-slim==4.1.3\n    # via fastmcp\nfilelock==4.2.4\n    # via\n    #   datasets\n    #   huggingface-hub\nflatbuffers==25.12.19\n    # via tensorflow\nflax==0.12.10\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   clu\n    #   gemma\n    #   kauldron\nfonttools==4.67.0\n    # via matplotlib\nfrozenlist==1.8.0\n    # via\n    #   aiohttp\n    #   aiosignal\nfsspec==2026.7.0\n    # via\n    #   datasets\n    #   etils\n    #   huggingface-hub\ngast==0.7.0\n    # via tensorflow\ngemma==4.0.1\n    # via -r /tmp/gemma-colab-requirements.in\ngitdb==4.0.12\n    # via gitpython\ngitpython==3.2.1\n    # via xmanager\ngoogle-api-core==2.42.0\n    # via\n    #   google-api-python-client\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   google-cloud-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage\n    #   xmanager\ngoogle-api-python-client==2.201.0\n    # via xmanager\ngoogle-auth==2.62.0\n    # via\n    #   cloud-sql-python-connector\n    #   google-api-core\n    #   google-api-python-client\n    #   google-auth-httplib2\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   google-cloud-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage\n    #   google-genai\n    #   xmanager\ngoogle-auth-httplib2==0.4.4\n    # via google-api-python-client\ngoogle-cloud-aiplatform==2.6.0\n    # via xmanager\ngoogle-cloud-bigquery==3.46.2\n    # via google-cloud-aiplatform\ngoogle-cloud-core==2.8.0\n    # via\n    #   google-cloud-bigquery\n    #   google-cloud-storage\ngoogle-cloud-resource-manager==1.19.0\n    # via google-cloud-aiplatform\ngoogle-cloud-sql==0.1.1\n    # via cloud-sql-python-connector\ngoogle-cloud-storage==3.17.1\n    # via\n    #   google-cloud-aiplatform\n    #   xmanager\ngoogle-crc32c==1.9.0\n    # via\n    #   google-cloud-storage\n    #   google-resumable-media\ngoogle-genai==2.32.0\n    # via google-cloud-aiplatform\ngoogle-pasta==0.2.0\n    # via tensorflow\ngoogle-resumable-media==2.11.0\n    # via\n    #   google-cloud-bigquery\n    #   google-cloud-storage\ngoogleapis-common-protos==1.75.5\n    # via\n    #   google-api-core\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   tensorflow-metadata\n    #   xmanager\ngrain==0.2.18\n    # via\n    #   gemma\n    #   kauldron\ngraphviz==0.21\n    # via kauldron\ngriffelib==2.4.0\n    # via fastmcp-slim\ngrpc-google-iam-v1==0.14.5\n    # via google-cloud-resource-manager\ngrpcio==1.84.0\n    # via\n    #   google-api-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   googleapis-common-protos\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   tensorboard\n    #   tensorflow\n    #   xmanager\ngrpcio-status==1.84.0\n    # via google-api-core\nh11==0.16.0\n    # via\n    #   httpcore\n    #   httpcore2\n    #   uvicorn\nh5py==3.14.0\n    # via\n    #   keras\n    #   tensorflow\nhf-xet==1.7.0\n    # via huggingface-hub\nhttpcore==1.0.9\n    # via httpx\nhttpcore2==2.16.0\n    # via httpx2\nhttplib2==0.32.0\n    # via\n    #   google-api-python-client\n    #   google-auth-httplib2\nhttpx==0.28.1\n    # via google-genai\nhttpx2==2.16.0\n    # via\n    #   fastmcp-slim\n    #   huggingface-hub\n    #   mcp\nhuggingface-hub==2.2.0\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   datasets\n    #   tokenizers\nhumanize==4.16.0\n    # via\n    #   orbax-checkpoint\n    #   xmanager\nidna==3.20\n    # via\n    #   anyio\n    #   email-validator\n    #   httpx\n    #   httpx2\n    #   requests\n    #   yarl\nimageio==2.38.1\n    # via scikit-image\nimmutabledict==4.3.1\n    # via\n    #   kauldron\n    #   tensorflow-datasets\n    #   xmanager\nipython==9.17.1\n    # via\n    #   ipywidgets\n    #   mediapy\nipython-pygments-lexers==1.1.1\n    # via ipython\nipywidgets==8.1.9\n    # via anywidget\njaraco-classes==3.4.0\n    # via keyring\njaraco-context==6.1.2\n    # via keyring\njaraco-functools==4.6.0\n    # via keyring\njax==0.12.0\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   chex\n    #   clu\n    #   flax\n    #   gemma\n    #   kauldron\n    #   optax\n    #   orbax-checkpoint\njaxlib==0.12.0\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   chex\n    #   clu\n    #   jax\n    #   optax\njaxtyping==0.3.11\n    # via\n    #   gemma\n    #   kauldron\njedi==0.20.1\n    # via ipython\njeepney==0.9.0\n    # via\n    #   keyring\n    #   secretstorage\njinja2==3.1.6\n    # via altair\njoblib==1.6.0\n    # via scikit-learn\njoserfc==1.7.7\n    # via\n    #   authlib\n    #   fastmcp-slim\njsonref==1.1.0\n    # via fastmcp-slim\njsonschema==4.26.0\n    # via\n    #   altair\n    #   mcp\njsonschema-path==0.5.0\n    # via fastmcp-slim\njsonschema-specifications==2025.9.1\n    # via jsonschema\njupyterlab-widgets==3.0.17\n    # via ipywidgets\nkauldron==1.4.2\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   gemma\nkeras==3.15.1\n    # via tensorflow\nkeyring==25.7.0\n    # via py-key-value-aio\nkiwisolver==1.5.1\n    # via matplotlib\nkubernetes==37.0.1\n    # via xmanager\nlark==1.3.1\n    # via\n    #   dialog\n    #   kauldron\nlazy-loader==0.6\n    # via scikit-image\nlibclang==18.1.1\n    # via tensorflow\nlibtpu==0.0.50\n    # via jax\nmako==1.4.3\n    # via alembic\nmarkdown==3.11.2\n    # via tensorboard\nmarkdown-it-py==4.2.0\n    # via rich\nmarkupsafe==3.0.4\n    # via\n    #   jinja2\n    #   mako\n    #   werkzeug\nmatplotlib==3.11.2\n    # via mediapy\nmatplotlib-inline==0.2.2\n    # via ipython\nmcp==2.3.0\n    # via\n    #   dialog\n    #   fastmcp-slim\n    #   gemma\nmcp-types==2.3.0\n    # via\n    #   fastmcp-slim\n    #   mcp\nmdurl==0.1.2\n    # via markdown-it-py\nmediapy==1.2.7\n    # via kauldron\nml-collections==1.1.0\n    # via\n    #   clu\n    #   kauldron\nml-dtypes==0.6.0\n    # via\n    #   jax\n    #   jaxlib\n    #   keras\n    #   tensorflow\n    #   tensorstore\nmore-itertools==11.2.1\n    # via\n    #   jaraco-classes\n    #   jaraco-functools\nmsgpack==1.2.3\n    # via\n    #   flax\n    #   orbax-checkpoint\nmultidict==7.1.0\n    # via\n    #   aiohttp\n    #   yarl\nmultiprocess==0.70.19\n    # via datasets\nnamex==0.1.0\n    # via keras\nnarwhals==2.27.1\n    # via\n    #   altair\n    #   scikit-learn\nnetworkx==3.7\n    # via scikit-image\nnumpy==2.5.4\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   chex\n    #   clu\n    #   contourpy\n    #   datasets\n    #   dialog\n    #   dm-tree\n    #   etils\n    #   flax\n    #   gemma\n    #   grain\n    #   h5py\n    #   imageio\n    #   jax\n    #   jaxlib\n    #   kauldron\n    #   keras\n    #   matplotlib\n    #   mediapy\n    #   ml-dtypes\n    #   opencv-python\n    #   optax\n    #   orbax-checkpoint\n    #   pandas\n    #   scikit-image\n    #   scikit-learn\n    #   scipy\n    #   tensorboard\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorstore\n    #   tifffile\n    #   treescope\noauthlib==4.0.0\n    # via requests-oauthlib\nopenapi-pydantic==0.6.0\n    # via fastmcp-slim\nopencv-python==5.0.0.93\n    # via kauldron\nopentelemetry-api==1.45.1\n    # via\n    #   fastmcp-slim\n    #   google-api-core\n    #   mcp\nopt-einsum==3.4.0\n    # via\n    #   jax\n    #   tensorflow\noptax==0.2.8\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   flax\n    #   kauldron\noptree==0.20.0\n    # via keras\norbax-checkpoint==0.12.8\n    # via\n    #   flax\n    #   gemma\n    #   kauldron\npackaging==26.3\n    # via\n    #   altair\n    #   clu\n    #   datasets\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   huggingface-hub\n    #   keras\n    #   lazy-loader\n    #   matplotlib\n    #   scikit-image\n    #   tensorboard\n    #   tensorflow\n    #   wheel\npandas==3.0.6\n    # via\n    #   datasets\n    #   kauldron\nparso==0.8.7\n    # via jedi\npathable==0.6.0\n    # via jsonschema-path\npexpect==4.9.0\n    # via ipython\npillow==13.0.0\n    # via\n    #   dialog\n    #   imageio\n    #   matplotlib\n    #   mediapy\n    #   scikit-image\n    #   tensorboard\nplatformdirs==4.13.5\n    # via fastmcp-slim\nportpicker==1.6.0\n    # via grain\nprometheus-client==0.26.0\n    # via orbax-checkpoint\npromise==2.3\n    # via tensorflow-datasets\nprompt-toolkit==3.0.53\n    # via ipython\npropcache==0.5.4\n    # via\n    #   aiohttp\n    #   yarl\nproto-plus==1.29.0\n    # via\n    #   google-api-core\n    #   google-cloud-aiplatform\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\nprotobuf==6.33.6\n    # via\n    #   google-api-core\n    #   google-cloud-aiplatform\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   googleapis-common-protos\n    #   grain\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   orbax-checkpoint\n    #   proto-plus\n    #   tensorboard\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorflow-metadata\n    #   xmanager\npsutil==7.2.2\n    # via\n    #   ipython\n    #   orbax-checkpoint\n    #   portpicker\n    #   tensorflow-datasets\n    #   xmanager\npsygnal==0.16.1\n    # via anywidget\nptyprocess==0.7.0\n    # via pexpect\npure-eval==0.2.4\n    # via stack-data\npy-key-value-aio==0.4.6\n    # via fastmcp-slim\npyarrow==26.0.0\n    # via\n    #   datasets\n    #   tensorflow-datasets\npyasn1==0.6.4\n    # via pyasn1-modules\npyasn1-modules==0.4.2\n    # via google-auth\npycparser==3.11\n    # via cffi\npydantic==2.14.1\n    # via\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   google-genai\n    #   kubernetes\n    #   mcp\n    #   mcp-types\n    #   openapi-pydantic\n    #   pydantic-settings\npydantic-core==2.50.1\n    # via pydantic\npydantic-settings==2.16.1\n    # via fastmcp-slim\npydub==0.25.1\n    # via dialog\npygments==2.21.0\n    # via\n    #   ipython\n    #   ipython-pygments-lexers\n    #   rich\n    #   rich-rst\npyjwt==2.15.1\n    # via mcp\npyparsing==3.3.3\n    # via\n    #   httplib2\n    #   matplotlib\npyperclip==1.11.0\n    # via fastmcp-slim\npython-dateutil==2.9.0.post0\n    # via\n    #   google-cloud-bigquery\n    #   kubernetes\n    #   matplotlib\n    #   pandas\npython-dotenv==1.2.4\n    # via\n    #   fastmcp-slim\n    #   pydantic-settings\n    #   xmanager\npython-multipart==0.0.32\n    # via\n    #   fastmcp-slim\n    #   mcp\npyyaml==6.0.3\n    # via\n    #   datasets\n    #   fastmcp-slim\n    #   flax\n    #   huggingface-hub\n    #   jsonschema-path\n    #   kubernetes\n    #   ml-collections\n    #   orbax-checkpoint\n    #   xmanager\nreferencing==0.37.0\n    # via\n    #   jsonschema\n    #   jsonschema-path\n    #   jsonschema-specifications\nrequests==2.34.2\n    # via\n    #   cloud-sql-python-connector\n    #   dialog\n    #   docker\n    #   google-api-core\n    #   google-auth\n    #   google-cloud-bigquery\n    #   google-cloud-storage\n    #   google-genai\n    #   jax\n    #   kubernetes\n    #   requests-oauthlib\n    #   tensorflow\n    #   tensorflow-datasets\n    #   xmanager\nrequests-oauthlib==2.0.0\n    # via kubernetes\nrich==15.0.0\n    # via\n    #   cyclopts\n    #   fastmcp-slim\n    #   flax\n    #   keras\n    #   rich-rst\nrich-rst==2.2.0\n    # via cyclopts\nrpds-py==2026.9.1\n    # via\n    #   jsonschema\n    #   referencing\nscikit-image==0.26.0\n    # via kauldron\nscikit-learn==1.9.1\n    # via kauldron\nscipy==1.18.1\n    # via\n    #   jax\n    #   jaxlib\n    #   scikit-image\n    #   scikit-learn\nsecretstorage==3.5.0\n    # via keyring\nsentencepiece==0.2.2\n    # via gemma\nsetuptools==84.2.1\n    # via\n    #   tensorboard\n    #   tensorflow\nsimple-parsing==0.1.9\n    # via tensorflow-datasets\nsimplejson==4.2.0\n    # via orbax-checkpoint\nsix==1.17.0\n    # via\n    #   astunparse\n    #   google-pasta\n    #   promise\n    #   python-dateutil\n    #   tensorflow\nsmmap==5.0.3\n    # via gitdb\nsniffio==1.3.1\n    # via google-genai\nsqlalchemy==2.1.5\n    # via\n    #   alembic\n    #   xmanager\nsqlparse==0.6.0\n    # via xmanager\nsse-starlette==3.5.0\n    # via mcp\nstack-data==0.6.3\n    # via ipython\nstarlette==1.9.0\n    # via\n    #   fastmcp-slim\n    #   mcp\n    #   sse-starlette\ntabulate==0.10.0\n    # via\n    #   kauldron\n    #   xmanager\ntenacity==9.1.4\n    # via google-genai\ntensorboard==2.21.0\n    # via kauldron\ntensorboard-data-server==0.7.2\n    # via tensorboard\ntensorflow==2.21.0\n    # via kauldron\ntensorflow-datasets==4.9.10\n    # via kauldron\ntensorflow-metadata==1.21.0\n    # via tensorflow-datasets\ntensorstore==0.1.85\n    # via\n    #   flax\n    #   orbax-checkpoint\ntermcolor==3.3.0\n    # via\n    #   tensorflow\n    #   tensorflow-datasets\n    #   xmanager\nthreadpoolctl==3.7.0\n    # via scikit-learn\ntifffile==2026.10.15\n    # via scikit-image\ntokenizers==0.23.3\n    # via -r /tmp/gemma-colab-requirements.in\ntoml==0.10.2\n    # via tensorflow-datasets\ntoolz==1.2.0\n    # via chex\ntqdm==4.70.1\n    # via\n    #   datasets\n    #   etils\n    #   huggingface-hub\n    #   kauldron\n    #   tensorflow-datasets\ntraitlets==5.16.1\n    # via\n    #   ipython\n    #   ipywidgets\n    #   matplotlib-inline\ntreescope==0.1.10\n    # via\n    #   flax\n    #   gemma\ntruststore==0.10.5\n    # via\n    #   httpcore2\n    #   httpx2\ntypeguard==4.6.0\n    # via kauldron\ntyping-extensions==4.16.0\n    # via\n    #   aiohttp\n    #   aiosignal\n    #   alembic\n    #   altair\n    #   anyio\n    #   anywidget\n    #   chex\n    #   clu\n    #   etils\n    #   exceptiongroup\n    #   fastmcp-slim\n    #   flax\n    #   google-cloud-aiplatform\n    #   google-genai\n    #   grpcio\n    #   httpx2\n    #   huggingface-hub\n    #   kauldron\n    #   kubernetes\n    #   mcp\n    #   mcp-types\n    #   opentelemetry-api\n    #   optree\n    #   orbax-checkpoint\n    #   py-key-value-aio\n    #   pydantic\n    #   pydantic-core\n    #   pydantic-settings\n    #   referencing\n    #   simple-parsing\n    #   sqlalchemy\n    #   starlette\n    #   tensorflow\n    #   typeguard\n    #   typing-inspection\ntyping-inspection==0.4.4\n    # via\n    #   mcp\n    #   pydantic\n    #   pydantic-settings\nuncalled-for==0.4.1\n    # via fastmcp-slim\nuritemplate==4.2.0\n    # via google-api-python-client\nurllib3==2.8.0\n    # via\n    #   docker\n    #   kubernetes\n    #   requests\nuvicorn==0.54.0\n    # via\n    #   fastmcp-slim\n    #   mcp\nuvloop==0.23.0\n    # via orbax-checkpoint\nwadler-lindig==0.1.7\n    # via jaxtyping\nwatchfiles==1.3.0\n    # via fastmcp-slim\nwcwidth==0.9.3\n    # via prompt-toolkit\nwebsocket-client==1.9.2\n    # via kubernetes\nwebsockets==16.1.1\n    # via\n    #   fastmcp-slim\n    #   google-genai\nwerkzeug==3.1.9\n    # via tensorboard\nwheel==0.48.0\n    # via astunparse\nwidgetsnbextension==4.0.16\n    # via ipywidgets\nwrapt==2.5.1\n    # via\n    #   clu\n    #   dm-tree\n    #   tensorflow\n    #   tensorflow-datasets\nxmanager==0.8.4\n    # via kauldron\nxxhash==4.0.2\n    # via datasets\nyarl==1.25.1\n    # via aiohttp\nzipp==4.1.1\n    # via etils\n\n\ntokamax @ git+https://github.com/openxla/tokamax.git@9ec839edafd361e316265e83d5b74dea01181ed5\n'
RESOLVED_DEPENDENCY_LOCK = '# This file was autogenerated by uv via the following command:\n#    uv pip compile /content/gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in --python /content/gemma_colab_pilot/.venv-V3/bin/python --output-file /content/gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.lock\nabsl-py==2.5.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   array-record\n    #   chex\n    #   clu\n    #   dm-tree\n    #   einshape\n    #   etils\n    #   gemma\n    #   grain\n    #   kauldron\n    #   keras\n    #   ml-collections\n    #   optax\n    #   orbax-checkpoint\n    #   qwix\n    #   tensorboard\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorflow-metadata\n    #   tokamax\n    #   xmanager\n    #   xprof\naiofile==3.12.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   py-key-value-aio\naiofiles==25.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   orbax-checkpoint\naiohappyeyeballs==2.7.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\naiohttp==3.14.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp-retry\n    #   cloud-sql-python-connector\n    #   fsspec\n    #   gcsfs\n    #   kubernetes\naiohttp-retry==2.9.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kubernetes\naiosignal==1.4.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\nalembic==1.20.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\naltair==6.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nannotated-types==0.8.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   pydantic\nanyio==4.15.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-genai\n    #   httpx\n    #   httpx2\n    #   mcp\n    #   py-key-value-aio\n    #   sse-starlette\n    #   starlette\n    #   watchfiles\nanywidget==0.11.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   dialog\narray-record==0.8.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   grain\n    #   tensorflow-datasets\nasttokens==3.0.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   stack-data\nastunparse==1.6.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\nasync-generator==1.10\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\nattrs==26.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\n    #   cyclopts\n    #   dm-tree\n    #   jsonschema\n    #   jsonschema-path\n    #   referencing\n    #   xmanager\nauthlib==1.9.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nbeartype==0.23.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   py-key-value-aio\ncachetools==7.2.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   py-key-value-aio\ncaio==0.12.9\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiofile\ncertifi==2026.10.19\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   httpcore\n    #   httpx\n    #   kubernetes\n    #   requests\ncffi==2.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cryptography\ncharset-normalizer==3.5.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   requests\ncheroot==11.1.2\n    # via xprof\nchex==0.1.92\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nclick==8.5.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   huggingface-hub\n    #   uvicorn\ncloud-sql-python-connector==1.23.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\ncloudpickle==3.1.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   grain\n    #   joblib\nclu==0.0.12\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ncomm==0.2.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipywidgets\ncontourpy==1.4.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   matplotlib\ncryptography==50.0.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   authlib\n    #   cloud-sql-python-connector\n    #   google-auth\n    #   joserfc\n    #   pyjwt\n    #   secretstorage\ncycler==0.12.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   matplotlib\ncyclopts==5.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\ndatasets==5.1.0\n    # via -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\ndecorator==5.3.1\n    # via gcsfs\ndialog==1.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\ndill==0.4.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   multiprocess\ndistro==1.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-genai\ndm-tree==0.1.10\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow-datasets\ndnspython==2.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cloud-sql-python-connector\n    #   email-validator\ndocker==7.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\ndocstring-parser==0.18.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cyclopts\n    #   google-cloud-aiplatform\n    #   simple-parsing\nduckdb==2.0.0\n    # via xprof\ndurationpy==0.11\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kubernetes\neinops==0.8.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   etils\n    #   gemma\n    #   kauldron\neinshape==1.0\n    # via tokamax\nemail-validator==2.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   pydantic\netils==1.14.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   array-record\n    #   clu\n    #   dialog\n    #   gemma\n    #   grain\n    #   kauldron\n    #   orbax-checkpoint\n    #   tensorflow-datasets\n    #   xmanager\n    #   xprof\nexceptiongroup==1.3.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nexecuting==2.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   stack-data\nfastmcp==4.1.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\nfastmcp-slim==4.1.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp\nfilelock==4.2.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   huggingface-hub\nfire==0.7.1\n    # via xprof\nflatbuffers==25.12.19\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\nflax==0.12.10\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   clu\n    #   gemma\n    #   kauldron\n    #   qwix\nfonttools==4.67.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   matplotlib\nfrozenlist==1.8.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\n    #   aiosignal\nfsspec==2026.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   etils\n    #   gcsfs\n    #   huggingface-hub\n    #   xprof\ngast==0.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\ngcsfs==2026.8.1\n    # via fsspec\ngemma==4.0.1\n    # via -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\ngitdb==4.0.12\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gitpython\ngitpython==3.2.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\ngoogle-api-core==2.42.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-python-client\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   google-cloud-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage\n    #   google-cloud-storage-control\n    #   xmanager\ngoogle-api-python-client==2.201.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\ngoogle-auth==2.62.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cloud-sql-python-connector\n    #   gcsfs\n    #   google-api-core\n    #   google-api-python-client\n    #   google-auth-httplib2\n    #   google-auth-oauthlib\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   google-cloud-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage\n    #   google-cloud-storage-control\n    #   google-genai\n    #   xmanager\ngoogle-auth-httplib2==0.4.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-python-client\ngoogle-auth-oauthlib==1.5.0\n    # via gcsfs\ngoogle-cloud-aiplatform==2.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\ngoogle-cloud-bigquery==3.46.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-aiplatform\ngoogle-cloud-core==2.8.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-bigquery\n    #   google-cloud-storage\ngoogle-cloud-resource-manager==1.19.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-aiplatform\ngoogle-cloud-sql==0.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cloud-sql-python-connector\ngoogle-cloud-storage==3.17.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gcsfs\n    #   google-cloud-aiplatform\n    #   xmanager\n    #   xprof\ngoogle-cloud-storage-control==1.16.0\n    # via gcsfs\ngoogle-crc32c==1.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-storage\n    #   google-resumable-media\ngoogle-genai==2.32.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-aiplatform\ngoogle-pasta==0.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\ngoogle-resumable-media==2.11.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-bigquery\n    #   google-cloud-storage\ngoogleapis-common-protos==1.75.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-core\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   tensorflow-metadata\n    #   xmanager\ngrain==0.2.18\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\n    #   kauldron\ngraphviz==0.21\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ngriffelib==2.4.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\ngrpc-google-iam-v1==0.14.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-resource-manager\n    #   google-cloud-storage-control\ngrpcio==1.84.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage-control\n    #   googleapis-common-protos\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   tensorboard\n    #   tensorflow\n    #   xmanager\ngrpcio-status==1.84.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-core\ngviz-api==1.10.0\n    # via xprof\nh11==0.16.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   httpcore\n    #   httpcore2\n    #   uvicorn\nh5py==3.14.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keras\n    #   tensorflow\nhf-xet==1.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   huggingface-hub\nhttpcore==1.0.9\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   httpx\nhttpcore2==2.16.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   httpx2\nhttplib2==0.32.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-python-client\n    #   google-auth-httplib2\nhttpx==0.28.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-genai\nhttpx2==2.16.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   huggingface-hub\n    #   mcp\nhuggingface-hub==2.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   tokenizers\nhumanize==4.16.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   orbax-checkpoint\n    #   xmanager\nidna==3.20\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   anyio\n    #   email-validator\n    #   httpx\n    #   httpx2\n    #   requests\n    #   yarl\nimageio==2.38.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-image\nimmutabledict==4.3.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\n    #   tensorflow-datasets\n    #   tokamax\n    #   xmanager\nipython==9.17.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipywidgets\n    #   mediapy\nipython-pygments-lexers==1.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\nipywidgets==8.1.9\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   anywidget\njaraco-classes==3.4.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keyring\njaraco-context==6.1.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keyring\njaraco-functools==4.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cheroot\n    #   keyring\njax==0.12.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   chex\n    #   clu\n    #   flax\n    #   gemma\n    #   kauldron\n    #   optax\n    #   orbax-checkpoint\n    #   qwix\n    #   tokamax\njaxlib==0.12.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   chex\n    #   clu\n    #   jax\n    #   optax\n    #   qwix\n    #   tokamax\njaxtyping==0.3.11\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\n    #   kauldron\n    #   tokamax\njedi==0.20.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\njeepney==0.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keyring\n    #   secretstorage\njinja2==3.1.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   altair\njoblib==1.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-learn\njoserfc==1.7.7\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   authlib\n    #   fastmcp-slim\njsonref==1.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\njsonschema==4.26.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   altair\n    #   mcp\njsonschema-path==0.5.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\njsonschema-specifications==2025.9.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jsonschema\njupyterlab-widgets==3.0.17\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipywidgets\nkauldron==1.4.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\nkeras==3.15.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\nkeyring==25.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   py-key-value-aio\nkiwisolver==1.5.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   matplotlib\nkubernetes==37.0.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\nlark==1.3.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   dialog\n    #   kauldron\nlazy-loader==0.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-image\nlibclang==18.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\nlibtpu==0.0.50\n    # via -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\nmako==1.4.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   alembic\nmarkdown==3.11.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorboard\nmarkdown-it-py==4.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   rich\nmarkupsafe==3.0.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jinja2\n    #   mako\n    #   werkzeug\nmatplotlib==3.11.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   mediapy\nmatplotlib-inline==0.2.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\nmcp==2.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   dialog\n    #   fastmcp-slim\n    #   gemma\nmcp-types==2.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   mcp\nmdurl==0.1.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   markdown-it-py\nmediapy==1.2.7\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nml-collections==1.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   clu\n    #   kauldron\nml-dtypes==0.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jax\n    #   jaxlib\n    #   keras\n    #   tensorflow\n    #   tensorstore\n    #   xprof\nmore-itertools==11.2.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cheroot\n    #   jaraco-classes\n    #   jaraco-functools\nmsgpack==1.2.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   flax\n    #   orbax-checkpoint\nmultidict==7.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\n    #   yarl\nmultiprocess==0.70.19\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\nnamex==0.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keras\nnarwhals==2.27.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   altair\n    #   scikit-learn\nnetworkx==3.7\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-image\nnumpy==2.5.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   chex\n    #   clu\n    #   contourpy\n    #   datasets\n    #   dialog\n    #   dm-tree\n    #   einshape\n    #   etils\n    #   flax\n    #   gemma\n    #   grain\n    #   h5py\n    #   imageio\n    #   jax\n    #   jaxlib\n    #   kauldron\n    #   keras\n    #   matplotlib\n    #   mediapy\n    #   ml-dtypes\n    #   opencv-python\n    #   optax\n    #   orbax-checkpoint\n    #   pandas\n    #   qwix\n    #   scikit-image\n    #   scikit-learn\n    #   scipy\n    #   tensorboard\n    #   tensorboardx\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorstore\n    #   tifffile\n    #   treescope\n    #   xprof\noauthlib==4.0.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   requests-oauthlib\nopenapi-pydantic==0.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nopencv-python==5.0.0.93\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nopentelemetry-api==1.45.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   google-api-core\n    #   mcp\nopt-einsum==3.4.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jax\n    #   qwix\n    #   tensorflow\noptax==0.2.8\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   flax\n    #   kauldron\noptree==0.20.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keras\norbax-checkpoint==0.12.8\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   flax\n    #   gemma\n    #   kauldron\npackaging==26.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   altair\n    #   clu\n    #   datasets\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   huggingface-hub\n    #   keras\n    #   lazy-loader\n    #   matplotlib\n    #   scikit-image\n    #   tensorboard\n    #   tensorboardx\n    #   tensorflow\n    #   wheel\npandas==3.0.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   kauldron\nparso==0.8.7\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jedi\npathable==0.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jsonschema-path\npexpect==4.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\npillow==13.0.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   dialog\n    #   imageio\n    #   matplotlib\n    #   mediapy\n    #   scikit-image\n    #   tensorboard\nplatformdirs==4.13.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nportpicker==1.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   grain\nprometheus-client==0.26.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   orbax-checkpoint\npromise==2.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow-datasets\nprompt-toolkit==3.0.53\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\npropcache==0.5.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\n    #   yarl\nproto-plus==1.29.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-core\n    #   google-cloud-aiplatform\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage-control\nprotobuf==6.33.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-core\n    #   google-cloud-aiplatform\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage-control\n    #   googleapis-common-protos\n    #   grain\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   orbax-checkpoint\n    #   proto-plus\n    #   tensorboard\n    #   tensorboardx\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorflow-metadata\n    #   xmanager\n    #   xprof\npsutil==7.2.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\n    #   orbax-checkpoint\n    #   portpicker\n    #   tensorflow-datasets\n    #   xmanager\npsygnal==0.16.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   anywidget\nptyprocess==0.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   pexpect\npure-eval==0.2.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   stack-data\npy-key-value-aio==0.4.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\npyarrow==26.0.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   tensorflow-datasets\npyasn1==0.6.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   pyasn1-modules\npyasn1-modules==0.4.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-auth\npycparser==3.11\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cffi\npydantic==2.14.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   google-genai\n    #   kubernetes\n    #   mcp\n    #   mcp-types\n    #   openapi-pydantic\n    #   pydantic-settings\n    #   tokamax\npydantic-core==2.50.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   pydantic\npydantic-settings==2.16.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\npydub==0.25.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   dialog\npygments==2.21.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\n    #   ipython-pygments-lexers\n    #   rich\n    #   rich-rst\npyjwt==2.15.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   mcp\npyparsing==3.3.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   httplib2\n    #   matplotlib\npyperclip==1.11.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\npython-dateutil==2.9.0.post0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-bigquery\n    #   kubernetes\n    #   matplotlib\n    #   pandas\npython-dotenv==1.2.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   pydantic-settings\n    #   xmanager\npython-multipart==0.0.32\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   mcp\npyyaml==6.0.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   fastmcp-slim\n    #   flax\n    #   huggingface-hub\n    #   jsonschema-path\n    #   kubernetes\n    #   ml-collections\n    #   orbax-checkpoint\n    #   xmanager\nqwix==0.1.8\n    # via tokamax\nreferencing==0.37.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jsonschema\n    #   jsonschema-path\n    #   jsonschema-specifications\nrequests==2.34.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cloud-sql-python-connector\n    #   dialog\n    #   docker\n    #   gcsfs\n    #   google-api-core\n    #   google-auth\n    #   google-cloud-bigquery\n    #   google-cloud-storage\n    #   google-genai\n    #   kubernetes\n    #   requests-oauthlib\n    #   tensorflow\n    #   tensorflow-datasets\n    #   xmanager\nrequests-oauthlib==2.0.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-auth-oauthlib\n    #   kubernetes\nrich==15.0.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cyclopts\n    #   fastmcp-slim\n    #   flax\n    #   keras\n    #   rich-rst\nrich-rst==2.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cyclopts\nrpds-py==2026.9.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jsonschema\n    #   referencing\nscikit-image==0.26.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nscikit-learn==1.9.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nscipy==1.18.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jax\n    #   jaxlib\n    #   scikit-image\n    #   scikit-learn\nsecretstorage==3.5.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keyring\nsentencepiece==0.2.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\nsetuptools==84.2.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorboard\n    #   tensorflow\nsimple-parsing==0.1.9\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow-datasets\nsimplejson==4.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   orbax-checkpoint\nsix==1.17.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   astunparse\n    #   google-pasta\n    #   gviz-api\n    #   promise\n    #   python-dateutil\n    #   tensorflow\n    #   xprof\nsmmap==5.0.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gitdb\nsniffio==1.3.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-genai\nsqlalchemy==2.1.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   alembic\n    #   xmanager\nsqlparse==0.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\nsse-starlette==3.5.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   mcp\nstack-data==0.6.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\nstarlette==1.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   mcp\n    #   sse-starlette\ntabulate==0.10.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\n    #   xmanager\ntenacity==9.1.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-genai\ntensorboard==2.21.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ntensorboard-data-server==0.7.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorboard\ntensorboardx==2.6.5\n    # via tokamax\ntensorflow==2.21.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ntensorflow-datasets==4.9.10\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ntensorflow-metadata==1.21.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow-datasets\ntensorstore==0.1.85\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   flax\n    #   orbax-checkpoint\ntermcolor==3.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fire\n    #   tensorflow\n    #   tensorflow-datasets\n    #   xmanager\nthreadpoolctl==3.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-learn\ntifffile==2026.10.15\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-image\ntokamax @ git+https://github.com/openxla/tokamax.git@9ec839edafd361e316265e83d5b74dea01181ed5\n    # via -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\ntokenizers==0.23.3\n    # via -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\ntoml==0.10.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow-datasets\ntoolz==1.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   chex\ntqdm==4.70.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   etils\n    #   huggingface-hub\n    #   kauldron\n    #   tensorflow-datasets\n    #   tokamax\ntraitlets==5.16.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\n    #   ipywidgets\n    #   matplotlib-inline\ntreescope==0.1.10\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   flax\n    #   gemma\ntruststore==0.10.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   httpcore2\n    #   httpx2\ntypeguard==4.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ntyping-extensions==4.16.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\n    #   aiosignal\n    #   alembic\n    #   altair\n    #   anyio\n    #   anywidget\n    #   chex\n    #   clu\n    #   etils\n    #   exceptiongroup\n    #   fastmcp-slim\n    #   flax\n    #   google-cloud-aiplatform\n    #   google-genai\n    #   grpcio\n    #   httpx2\n    #   huggingface-hub\n    #   kauldron\n    #   kubernetes\n    #   mcp\n    #   mcp-types\n    #   opentelemetry-api\n    #   optree\n    #   orbax-checkpoint\n    #   py-key-value-aio\n    #   pydantic\n    #   pydantic-core\n    #   pydantic-settings\n    #   referencing\n    #   simple-parsing\n    #   sqlalchemy\n    #   starlette\n    #   tensorflow\n    #   typeguard\n    #   typing-inspection\ntyping-inspection==0.4.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   mcp\n    #   pydantic\n    #   pydantic-settings\nuncalled-for==0.4.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nuritemplate==4.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-python-client\nurllib3==2.8.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   docker\n    #   kubernetes\n    #   requests\n    #   xprof\nuvicorn==0.54.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   mcp\nuvloop==0.23.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   orbax-checkpoint\nwadler-lindig==0.1.7\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jaxtyping\nwatchfiles==1.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nwcwidth==0.9.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   prompt-toolkit\nwebsocket-client==1.9.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kubernetes\nwebsockets==16.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   google-genai\nwerkzeug==3.1.9\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorboard\n    #   xprof\nwheel==0.48.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   astunparse\nwidgetsnbextension==4.0.16\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipywidgets\nwrapt==2.5.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   clu\n    #   dm-tree\n    #   tensorflow\n    #   tensorflow-datasets\nxmanager==0.8.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nxprof==2.24.0\n    # via tokamax\nxxhash==4.0.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\nyarl==1.25.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\nzipp==4.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   etils\n'
import os
import hashlib
import sys
import json
import time
import uuid
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

NOTEBOOK_START_MONOTONIC = time.monotonic()
BASE = Path("/content/gemma_colab_pilot")
BASE.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
WORK = BASE / RUN_ID
WORK.mkdir(exist_ok=False)
if shutil.disk_usage(BASE).free < 4 * 1024**3:
    raise RuntimeError("This weights-only pilot needs at least 4 GiB free local disk.")
from google.colab import drive
drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/ShiningGem/colab-v5e1/V3.6") / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)
DEPENDENCY_CACHE = OUTPUT.parent.parent / "V3-dependencies"
DEPENDENCY_CACHE.mkdir(parents=True, exist_ok=True)

def setup_command(command, log_name, environment=None):
    with (OUTPUT / log_name).open("x") as log:
        process = subprocess.Popen(command, env=environment, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        status = process.wait()
    if status:
        raise RuntimeError(f"Setup failed (exit {status}); error above and in {OUTPUT / log_name}.")

INPUT_PATH = WORK / "requirements.in"
LOCK_PATH = WORK / "requirements.lock"
INPUT_PATH.write_text(REQUIREMENTS_INPUT)
LOCK_PATH.write_text(RESOLVED_DEPENDENCY_LOCK)
LOCK_DIGEST = hashlib.sha256(LOCK_PATH.read_bytes()).hexdigest()
INPUT_DIGEST = hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest()
TOOLING = BASE / ".bootstrap"
if not (TOOLING / "uv").is_dir():
    setup_command([sys.executable, "-m", "pip", "install", "--quiet", "--target", str(TOOLING),
                   "uv==0.11.15"], "setup-bootstrap.log")
uv_env = dict(os.environ, PYTHONPATH=str(TOOLING))
uv = [sys.executable, "-m", "uv"]
PYTHON = BASE / ".venv-V3" / "bin" / "python"
if not PYTHON.exists():
    setup_command([*uv, "venv", "--python", "3.12.15", "--managed-python",
                   str(PYTHON.parent.parent)], "setup-python.log", uv_env)
ENV_MARKER = PYTHON.parent.parent / ".v3-lock-sha256"
ENVIRONMENT_REUSED = ENV_MARKER.exists() and ENV_MARKER.read_text().strip() == LOCK_DIGEST
if ENVIRONMENT_REUSED:
    print("Reusing matching V3.4 environment; no package resolution or install.", flush=True)
else:
    print("Installing the embedded successful V3.4 lock; no dependency resolution step.", flush=True)
    setup_command([*uv, "pip", "install", "--quiet", "--python", str(PYTHON),
                   "-r", str(LOCK_PATH)], "setup-install.log", uv_env)
    ENV_MARKER.write_text(LOCK_DIGEST + "\n")
PERSISTENT_JAX_CACHE = OUTPUT.parent.parent / "V3-cache" / "jax"
if PERSISTENT_JAX_CACHE.exists():
    shutil.copytree(PERSISTENT_JAX_CACHE, BASE / "jax-cache-V3", dirs_exist_ok=True)
SETUP_SECONDS = time.monotonic() - NOTEBOOK_START_MONOTONIC
shutil.copyfile(INPUT_PATH, OUTPUT / "requirements.in")
shutil.copyfile(LOCK_PATH, OUTPUT / "requirements.lock")
(OUTPUT / "dependency_source.json").write_text(json.dumps({
    "requirements_input_sha256": INPUT_DIGEST, "requirements_lock_sha256": LOCK_DIGEST,
    "source": "successful user-supplied V3.4 report lock", "environment_reused": ENVIRONMENT_REUSED,
    "dependency_resolution_performed": False, "bootstrap_uv": "0.11.15",
    "tokamax_execution": False, "splash_execution": False}, indent=2))
print(f"Environment ready in {SETUP_SECONDS:.1f} seconds.")
print("Results:", OUTPUT)


## 2. Authorize the tokenizer
The default is fixed. TOKENIZER_REFERENCE is the only research input to change for later tokenizer comparisons.


In [ ]:
from google.colab import userdata
from getpass import getpass

TOKENIZER_REFERENCE = "google/gemma-3-270m" #@param {type:"string"}
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token with Gemma repository access: ")
if not HF_TOKEN.strip():
    raise ValueError("A Hugging Face token is required for the default gated tokenizer.")


## 3. Write the embedded program
The next cell writes source and configuration; it does not execute training.


In [ ]:
PROGRAM_SOURCE = '"""Gemma Colab V3.6: one fixed multilingual tokenizer-screening pilot on one v5e TPU.\n"""\nfrom __future__ import annotations\nimport argparse\nimport dataclasses\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport time\nimport traceback\nimport zipfile\nimport numpy as np\n\ndef read_json(path):\n    return json.loads(Path(path).read_text())\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\\n")\n    temporary.replace(path)\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef chunk_document(ids, sequence_length, bos_id, eos_id, pad_id=0):\n    """Every target appears once; consecutive chunks overlap by one input token."""\n    if pad_id in ids:\n        raise ValueError("Document encodes a padding ID; cannot silently mask real text.")\n    sequence = [bos_id, *ids, eos_id]\n    for start in range(0, len(sequence) - 1, sequence_length):\n        window = sequence[start:start + sequence_length + 1]\n        count = len(window) - 1\n        tokens = np.full(sequence_length, pad_id, np.int32)\n        targets = np.full(sequence_length, pad_id, np.int32)\n        mask = np.zeros(sequence_length, np.bool_)\n        tokens[:count] = window[:-1]\n        targets[:count] = window[1:]\n        mask[:count] = True\n        yield tokens, targets, mask\n\n\ndef accelerator_imports(cpu_test=False):\n    global jax, jnp, optax, struct, serialization, traverse_util\n    os.environ["JAX_PLATFORMS"] = "cpu" if cpu_test else "tpu"\n    import jax\n    import jax.numpy as jnp\n    import optax\n    from flax import struct, serialization, traverse_util\n    return jax\n\n\ndef preflight(cpu_test=False):\n    accelerator_imports(cpu_test)\n    devices = jax.devices()\n    if len(devices) != 1:\n        raise RuntimeError(f"This notebook requires exactly ONE v5e chip; found {len(devices)} devices: {devices}. Select the v5e-1 TPU runtime in Colab.")\n    device = devices[0]\n    if not cpu_test:\n        kind = device.device_kind.lower().replace(" ", "")\n        if device.platform != "tpu" or not any(label in kind for label in ("v5lite", "v5e")):\n            raise RuntimeError(f"Expected a v5e-1 TPU, found {device}. Change the Colab runtime hardware accelerator.")\n    return {"platform": device.platform, "device_kind": device.device_kind,\n            "device_count": len(devices), "cpu_test": cpu_test,\n            "python": sys.version, "jax": jax.__version__,\n            "packages": {d.metadata["Name"]: d.version for d in importlib.metadata.distributions()}}\n\n\ndef numpy_leaf(value):\n    if isinstance(value, jax.Array):\n        return np.asarray(jax.device_get(value))\n    return np.asarray(value)\n\n\ndef archive_tree(state_dict):\n    leaves = []\n    def encode(node):\n        if isinstance(node, dict):\n            return {key: encode(value) for key, value in node.items()}\n        name = f"arrays/{len(leaves):05d}.npy"\n        leaves.append((name, node))\n        return {"__array__": name}\n    return encode(state_dict), leaves\n\n\ndef save_archive(path, state, metadata):\n    """Write one array at a time; never materialize a full host checkpoint copy."""\n    tree, leaves = archive_tree(serialization.to_state_dict(state))\n    temporary = Path(str(path) + ".partial")\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:\n        for name, leaf in leaves:\n            array = numpy_leaf(leaf)\n            with archive.open(name, "w", force_zip64=True) as stream:\n                np.lib.format.write_array(stream, array, allow_pickle=False)\n            del array\n        archive.writestr("checkpoint.json", json.dumps({"tree": tree, "metadata": metadata}, allow_nan=False))\n    temporary.replace(path)\n    return len(leaves)\n\n\ndef restore_archive(path, abstract_state):\n    """Restore arrays directly to the TPU before reconstructing the state class."""\n    with zipfile.ZipFile(path) as archive:\n        description = json.loads(archive.read("checkpoint.json"))\n        def decode(node):\n            if set(node) == {"__array__"}:\n                with archive.open(node["__array__"]) as stream:\n                    array = np.lib.format.read_array(stream, allow_pickle=False)\n                value = jax.device_put(array)\n                jax.block_until_ready(value)\n                return value\n            return {key: decode(value) for key, value in node.items()}\n        restored_dict = decode(description["tree"])\n    return serialization.from_state_dict(abstract_state, restored_dict)\n\n\nMODEL_HELPERS = \'@struct.dataclass\\nclass PilotState:\\n    step: jax.Array\\n    params: object\\n    opt_state: object\\n    rng: jax.Array\\n    data_position: jax.Array\\n\'\n\ndef pack_chunks(arrays, seed):\n    """Best-fit decreasing, preserving each original chunk\'s complete context."""\n    tokens, targets, masks = (arrays[key] for key in ("tokens", "targets", "mask"))\n    count, length = tokens.shape\n    lengths = masks.sum(axis=1).astype(int)\n    if np.any(lengths <= 0) or not np.array_equal(masks, np.arange(length)[None, :] < lengths[:, None]):\n        raise ValueError("Input chunks must have a nonempty contiguous prefix of scored targets.")\n    shuffled = np.random.default_rng(seed).permutation(count)\n    ordered = shuffled[np.argsort(-lengths[shuffled], kind="stable")]\n    buckets = [[] for _ in range(length + 1)]\n    bins, assignments = [], []\n    for source in ordered:\n        size = int(lengths[source])\n        free = next((room for room in range(size, length + 1) if buckets[room]), None)\n        if free is None:\n            index = len(bins)\n            bins.append(0)\n            free = length\n        else:\n            index = buckets[free].pop()\n        start = bins[index]\n        bins[index] += size\n        buckets[free - size].append(index)\n        assignments.append([int(source), index, start, size])\n    shape = (len(bins), length)\n    packed = {"tokens": np.zeros(shape, np.int32), "targets": np.zeros(shape, np.int32),\n              "mask": np.zeros(shape, np.bool_), "positions": np.zeros(shape, np.int32),\n              "segment_ids": np.full(shape, -1, np.int32)}\n    seen = np.zeros(count, np.int32)\n    for source, index, start, size in assignments:\n        sl = slice(start, start + size)\n        packed["tokens"][index, sl] = tokens[source, :size]\n        packed["targets"][index, sl] = targets[source, :size]\n        packed["mask"][index, sl] = True\n        packed["positions"][index, sl] = np.arange(size)\n        packed["segment_ids"][index, sl] = source\n        seen[source] += 1\n    # Verify content and positions before discarding the unpacked training arrays.\n    for source, index, start, size in assignments:\n        sl = slice(start, start + size)\n        if not (np.array_equal(packed["tokens"][index, sl], tokens[source, :size])\n                and np.array_equal(packed["targets"][index, sl], targets[source, :size])\n                and np.array_equal(packed["positions"][index, sl], np.arange(size))):\n            raise ValueError("Packing changed a source chunk.")\n    if not np.all(seen == 1) or int(packed["mask"].sum()) != int(lengths.sum()):\n        raise ValueError("Packing duplicated or dropped source targets.")\n    report = {"version": PACKING_VERSION, "source_chunks": count, "packed_sequences": len(bins),\n              "scored_tokens": int(lengths.sum()), "positions": int(np.prod(shape)),\n              "occupancy": float(lengths.sum() / np.prod(shape)),\n              "targets_preserved": True, "context_preserved": True,\n              "assignments_source_bin_start_length": assignments}\n    return packed, report\n\n\ndef packed_attention_mask(segment_ids):\n    length = segment_ids.shape[-1]\n    same_segment = segment_ids[:, :, None] == segment_ids[:, None, :]\n    causal = jnp.arange(length)[:, None] >= jnp.arange(length)[None, :]\n    valid = segment_ids >= 0\n    allowed = same_segment & causal[None, :, :] & valid[:, :, None] & valid[:, None, :]\n    # Padding queries attend to themselves so every softmax has at least one key.\n    return allowed | ((~valid)[:, :, None] & jnp.eye(length, dtype=jnp.bool_)[None, :, :])\n\n\ndef batch_training_arrays(arrays, batch_size, seed):\n    order = np.random.default_rng(seed).permutation(len(arrays["tokens"]))\n    padded = (-len(order)) % batch_size\n    result = {}\n    for key, values in arrays.items():\n        values = values[order]\n        if padded:\n            values = np.pad(values, ((0, padded), (0, 0)), constant_values=-1 if key == "segment_ids" else 0)\n        result[key] = values.reshape((-1, batch_size, values.shape[-1]))\n    return result, hashlib.sha256(order.astype("<i8").tobytes()).hexdigest()\n\n\n\n"""V3 extensions, concatenated with base.py by build_notebook.py; no local execution."""\nimport functools\nfrom collections import Counter, defaultdict\nfrom urllib.parse import urlsplit, urlunsplit\n\nPROTOCOL = "V3-static-byte-mix-1"\nPACKING_VERSION = "V3-intact-chunks-1"\nLANGUAGES = ["eng_Latn", "fra_Latn", "spa_Latn", "por_Latn", "cmn_Hani", "arb_Arab",\n             "hin_Deva", "swh_Latn", "zul_Latn", "xho_Latn", "tsn_Latn", "sot_Latn"]\nWIKI_CODES = ["en", "fr", "es", "pt", "zh", "ar", "hi", "sw", "zu", "xh", "tn", "st"]\nDOMAINS = ["pdf_edu", "dclm", "web_edu", "multilingual_web", "math", "code", "translations", "wiki_holdout"]\nSMOL = "HuggingFaceFW/finepdfs_edu_50BT-dclm_30BT-fineweb_edu_20BT-shuffled"\nSOURCE_REPOS = [SMOL, "HuggingFaceFW/fineweb-2", "HuggingFaceTB/finemath",\n                "codeparrot/codeparrot-clean", "HuggingFaceFW/finetranslations", "HuggingFaceFW/finewiki"]\nINPUT_KEYS = ("tokens", "targets", "mask", "positions", "segment_ids", "text_mask", "byte_weights", "language_ids", "domain_ids")\nEXTRA_KEYS = ("text_mask", "byte_weights", "language_ids", "domain_ids")\n\n\ndef recipe():\n    # Basis points of original UTF-8 bytes, not tokenizer-dependent token counts.\n    multilingual = [10, 10, 8, 15, 15, 15, 8, 5, 5, 5, 4]\n    buckets = [dict(key="pdf_edu", repo=SMOL, subset=None, language="eng_Latn", domain="pdf_edu", bp=2750),\n               dict(key="dclm", repo=SMOL, subset=None, language="eng_Latn", domain="dclm", bp=1650),\n               dict(key="web_edu", repo=SMOL, subset=None, language="eng_Latn", domain="web_edu", bp=1100)]\n    buckets += [dict(key="fw2_" + lang, repo=SOURCE_REPOS[1], subset=lang, language=lang,\n                     domain="multilingual_web", bp=30 * share) for lang, share in zip(LANGUAGES[1:], multilingual)]\n    buckets += [dict(key="math", repo=SOURCE_REPOS[2], subset="finemath-3plus", language="eng_Latn", domain="math", bp=500),\n                dict(key="code", repo=SOURCE_REPOS[3], subset=None, language="eng_Latn", domain="code", bp=500)]\n    buckets += [dict(key="translations_" + lang, repo=SOURCE_REPOS[4], subset=lang, language=lang,\n                     domain="translations", bp=125) for lang in ("fra_Latn", "arb_Arab", "hin_Deva", "swh_Latn")]\n    return {"protocol": PROTOCOL, "training_bytes_target": 64 * 1024**2,\n            "in_mix_eval_bytes_target": 256 * 1024, "wiki_eval_bytes_per_language": 64 * 1024,\n            "max_document_span_bytes": 32 * 1024, "byte_quota_tolerance": 3,\n            "source_weight_unit": "original UTF-8 bytes", "buckets": buckets,\n            "split_rule": "SHA256(source group) modulo 10: 0 held out, 1..9 train",\n            "sampling_scope": "bounded deterministic shuffled stream prefixes; not a uniform full-corpus sample"}\n\n\ndef protocol_fingerprint():\n    return hashlib.sha256(json.dumps(recipe(), sort_keys=True).encode()).hexdigest()\n\n\ndef canonical_group(row, text):\n    metadata = row.get("metadata")\n    url = row.get("url") or (metadata.get("url") if isinstance(metadata, dict) else None)\n    if url:\n        parsed = urlsplit(str(url))\n        return "url:" + urlunsplit(("", parsed.netloc.lower().removeprefix("www."), parsed.path.rstrip("/"), parsed.query, ""))\n    if row.get("repo_name"):\n        return "code-repo:" + str(row["repo_name"])\n    return "id:" + str(row.get("id") or row.get("blob_id") or hashlib.sha256(text.encode()).hexdigest())\n\n\ndef utf8_prefix(text, limit):\n    return text.encode("utf-8")[:limit].decode("utf-8", errors="ignore")\n\n\ndef read_jsonl(path):\n    with Path(path).open(encoding="utf-8") as stream:\n        return [json.loads(line) for line in stream]\n\n\ndef verify_raw(directory):\n    manifest = read_json(directory / "manifest.json")\n    if manifest["recipe_sha256"] != protocol_fingerprint():\n        raise ValueError("The shared raw corpus belongs to another V3 protocol.")\n    for name, digest in manifest["files"].items():\n        if sha256_file(directory / name) != digest:\n            raise ValueError(f"Raw corpus checksum differs: {name}")\n    return manifest\n\n\n\n"""V3.6 parallel source preparation. Embedded in the notebook; not run locally."""\nfrom concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED\nimport threading\n\nDOWNLOAD_WORKERS = 6\nPREFIX_CACHE_VERSION = "v31-source-prefix-1"\nPREFIX_FACTOR = 1.25\nPREFIX_PART_BYTES = 4 * 1024**2\nPREFIX_JOB_LIMIT_BYTES = 256 * 1024**2\n\n\ndef immutable_sources(shared):\n    from huggingface_hub import HfApi\n    path = shared / "source-lock.json"\n    if path.exists():\n        return read_json(path)\n    def resolve(repo):\n        return HfApi().dataset_info(repo, revision="main").sha\n    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:\n        revisions = list(pool.map(resolve, SOURCE_REPOS))\n    lock = dict(zip(SOURCE_REPOS, revisions))\n    write_json(path, lock)\n    return lock\n\n\ndef source_jobs(lock):\n    from datasets import get_dataset_config_names\n    rules = recipe()\n    wiki_repo = SOURCE_REPOS[-1]\n    names = set(get_dataset_config_names(wiki_repo, revision=lock[wiki_repo]))\n    wiki = []\n    for language, code in zip(LANGUAGES, WIKI_CODES):\n        subset = next((name for name in (code, code + "wiki") if name in names), None)\n        if subset is None:\n            raise ValueError(f"Pinned FineWiki has no configuration for {language}.")\n        wiki.append({"key": "wiki_" + language, "kind": "wiki", "repo": wiki_repo,\n            "subset": subset, "revision": lock[wiki_repo], "language": language,\n            "targets": {"wiki:wiki_" + language: rules["wiki_eval_bytes_per_language"]}, "row_limit": 100000})\n    grouped = defaultdict(list)\n    for bucket in rules["buckets"]:\n        grouped[(bucket["repo"], bucket["subset"])].append(bucket)\n    training = []\n    for (repo, subset), buckets in grouped.items():\n        targets = {}\n        for bucket in buckets:\n            targets["train:" + bucket["key"]] = round(rules["training_bytes_target"] * bucket["bp"] / 10000)\n            targets["in_mix:" + bucket["key"]] = round(rules["in_mix_eval_bytes_target"] * bucket["bp"] / 10000)\n        training.append({"key": "smol_english" if repo == SMOL else buckets[0]["key"], "kind": "training",\n            "repo": repo, "subset": subset, "revision": lock[repo], "buckets": buckets,\n            "targets": targets, "row_limit": 300000})\n    # Canonical merge order is V3 order, independent of thread completion order.\n    return wiki + training\n\n\ndef candidate_records(job, row):\n    rules = recipe()\n    if job["kind"] == "wiki":\n        texts = [(row.get("text"), job["language"])]\n        bucket_key, domain = job["key"], "wiki_holdout"\n    else:\n        if row.get("early_stop") or row.get("is_truncated"):\n            return []\n        if job["repo"] == SMOL:\n            origin = str(row.get("dataset", ""))\n            bucket_key = "pdf_edu" if "finepdfs" in origin else "web_edu" if "fineweb-edu" in origin else "dclm" if "dclm" in origin else None\n            if bucket_key is None:\n                raise ValueError("Unknown upstream Smol-Data source label: " + origin)\n            bucket = next(bucket for bucket in job["buckets"] if bucket["key"] == bucket_key)\n        else:\n            bucket = job["buckets"][0]\n            bucket_key = bucket["key"]\n        domain = bucket["domain"]\n        if job["repo"] == SOURCE_REPOS[4]:\n            score = row.get("edu_score")\n            if score is None or int(score) < 2:\n                return []\n            texts = [(row.get("og_full_text"), bucket["language"]), (row.get("translated_text"), "eng_Latn")]\n        else:\n            texts = [(row.get("content") if domain == "code" else row.get("text"), bucket["language"])]\n    group = canonical_group(row, texts[0][0] or "")\n    if job["kind"] != "wiki" and "wikipedia.org" in group:\n        return []\n    split = "wiki" if job["kind"] == "wiki" else "in_mix" if int(hashlib.sha256(group.encode()).hexdigest()[:16], 16) % 10 == 0 else "train"\n    provenance = {key: row.get(key) for key in ("id", "url", "repo_name", "path", "license", "dataset") if row.get(key) is not None}\n    candidates = []\n    for text, language in texts:\n        if not isinstance(text, str) or not text.strip():\n            continue\n        encoded = text.encode("utf-8")\n        span = utf8_prefix(text, rules["max_document_span_bytes"])\n        candidates.append({"text": span, "original_document_sha256": hashlib.sha256(encoded).hexdigest(),\n            "source_was_clipped": span != text, "source_group": group, "source": provenance,\n            "language": language, "domain": domain, "bucket": bucket_key, "split": split,\n            "utf8_bytes": len(span.encode("utf-8"))})\n    return candidates\n\n\ndef materialize_prefix(job, root, factor, report_status, stop):\n    """Cache deterministic source prefixes in independently committed 4 MiB target batches."""\n    from datasets import load_dataset, DownloadConfig\n    started = time.monotonic()\n    job_hash = hashlib.sha256(json.dumps({"version": PREFIX_CACHE_VERSION, "job": job}, sort_keys=True).encode()).hexdigest()\n    directory = root / job_hash[:24]\n    directory.mkdir(parents=True, exist_ok=True)\n    state_path = directory / "state.json"\n    state = read_json(state_path) if state_path.exists() else {"job_sha256": job_hash, "job": job,\n        "rows_read": 0, "candidate_bytes": {}, "candidate_documents": {}, "parts": [], "exhausted": False}\n    if state["job_sha256"] != job_hash:\n        raise ValueError("Source-prefix cache identity mismatch: " + job["key"])\n    for part in state["parts"]:\n        if sha256_file(directory / part["name"]) != part["sha256"]:\n            raise ValueError("Source-prefix cache checksum mismatch: " + job["key"])\n    original_rows = state["rows_read"]\n    original_parts = len(state["parts"])\n    desired = {key: math.ceil(value * factor) for key, value in job["targets"].items()}\n    def complete():\n        return all(state["candidate_bytes"].get(key, 0) >= quota and state["candidate_documents"].get(key, 0) >= 4\n                   for key, quota in desired.items())\n    def status(stage):\n        report_status(job["key"], {"stage": stage, "rows": state["rows_read"],\n            "candidate_bytes": sum(state["candidate_bytes"].values()), "seconds": time.monotonic() - started})\n    def result():\n        return {"directory": str(directory), "rows_read": state["rows_read"], "candidate_parts": len(state["parts"]),\n            "candidate_bytes": sum(state["candidate_bytes"].values()), "prefix_cache_reused": original_parts > 0,\n            "new_rows_read": state["rows_read"] - original_rows, "seconds": time.monotonic() - started}\n    if complete() or state["exhausted"]:\n        status("cached")\n        return result()\n    buffer = bytearray()\n    committed_bytes = sum(part["bytes"] for part in state["parts"])\n    last_flush = time.monotonic()\n    def commit():\n        nonlocal buffer, last_flush, committed_bytes\n        if buffer:\n            name = f"part-{len(state[\'parts\']):05d}.jsonl"\n            temporary = directory / (name + ".partial")\n            temporary.write_bytes(buffer)\n            temporary.replace(directory / name)\n            state["parts"].append({"name": name, "bytes": len(buffer), "sha256": hashlib.sha256(buffer).hexdigest()})\n            committed_bytes += len(buffer)\n            buffer = bytearray()\n        # A cursor is committed only after all candidates from its row are committed.\n        write_json(state_path, state)\n        last_flush = time.monotonic()\n    status("opening stream")\n    kwargs = {"name": job["subset"], "split": "train", "streaming": True, "revision": job["revision"],\n              "download_config": DownloadConfig(max_retries=1)}\n    if job["repo"] == SOURCE_REPOS[4]:\n        # Avoid downloading duplicate original/translated chunk-list columns.\n        kwargs["columns"] = ["id", "url", "og_full_text", "translated_text", "og_language", "edu_score", "early_stop"]\n    try:\n        stream = load_dataset(job["repo"], **kwargs)\n        if state["rows_read"]:\n            stream = stream.skip(state["rows_read"])\n        status("reading")\n        for row in stream:\n            if stop.is_set():\n                commit()\n                status("stopped; cache saved")\n                return result()\n            if state["rows_read"] >= job["row_limit"]:\n                state["exhausted"] = True\n                break\n            records = candidate_records(job, row)\n            encoded_records = [(json.dumps(record, ensure_ascii=False) + "\\n").encode("utf-8") for record in records]\n            if committed_bytes + len(buffer) + sum(map(len, encoded_records)) > PREFIX_JOB_LIMIT_BYTES:\n                raise RuntimeError("Source-prefix cache would exceed its 256 MiB bound: " + job["key"])\n            for record, encoded_record in zip(records, encoded_records):\n                buffer.extend(encoded_record)\n                stratum = record["split"] + ":" + record["bucket"]\n                state["candidate_bytes"][stratum] = state["candidate_bytes"].get(stratum, 0) + record["utf8_bytes"]\n                state["candidate_documents"][stratum] = state["candidate_documents"].get(stratum, 0) + 1\n            state["rows_read"] += 1\n            if len(buffer) >= PREFIX_PART_BYTES or time.monotonic() - last_flush >= 5:\n                commit()\n                status("reading")\n            if complete():\n                break\n        else:\n            state["exhausted"] = True\n        commit()\n        status("ready")\n        return result()\n    except Exception:\n        commit()\n        status("failed; cache saved")\n        raise\n\n\ndef build_raw_corpus(destination, lock):\n    started = time.monotonic()\n    destination.mkdir(parents=True, exist_ok=True)\n    if (destination / "manifest.json").exists():\n        # A previous interrupted run may have completed the final corpus already.\n        manifest = verify_raw(destination)\n        write_json(destination.parent / "parallel_downloads.json", {"version": "V3.6",\n            "recovered_complete_staging": True, "max_workers": DOWNLOAD_WORKERS,\n            "peak_active_source_jobs": 0, "source_jobs": 0, "fetch_seconds": 0.,\n            "note": "Verified complete staged corpus recovered without source fetching."})\n        return manifest\n    shared = destination.parent\n    cache_root = shared / PREFIX_CACHE_VERSION\n    cache_root.mkdir(parents=True, exist_ok=True)\n    jobs = source_jobs(lock)\n    status_lock, stop = threading.Lock(), threading.Event()\n    statuses = {job["key"]: {"stage": "queued", "rows": 0, "candidate_bytes": 0, "seconds": 0.} for job in jobs}\n    peak_active = 0\n    def status(key, value):\n        nonlocal peak_active\n        with status_lock:\n            statuses[key] = value\n            active = sum(item["stage"] in ("opening stream", "reading") for item in statuses.values())\n            peak_active = max(peak_active, active)\n    # Start the large English stream and all four translations early; interleave Wiki/FW2 thereafter.\n    front = [job for job in jobs if job["key"] == "smol_english" or job["repo"] == SOURCE_REPOS[4]]\n    wiki_jobs = [job for job in jobs if job["kind"] == "wiki"]\n    other_jobs = [job for job in jobs if job not in front and job["kind"] != "wiki"]\n    scheduled = front[:]\n    for index in range(max(len(wiki_jobs), len(other_jobs))):\n        if index < len(wiki_jobs):\n            scheduled.append(wiki_jobs[index])\n        if index < len(other_jobs):\n            scheduled.append(other_jobs[index])\n    print(f"V3.6: {len(jobs)} source jobs; {DOWNLOAD_WORKERS} concurrent workers; completed batches persist in Drive.", flush=True)\n    reports = {}\n    pool = ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS)\n    futures = {pool.submit(materialize_prefix, job, cache_root, PREFIX_FACTOR, status, stop): job for job in scheduled}\n    pending = set(futures)\n    try:\n        while pending:\n            done, pending = wait(pending, timeout=5, return_when=FIRST_COMPLETED)\n            for future in done:\n                job = futures[future]\n                try:\n                    report = future.result()\n                except Exception:\n                    write_json(shared / "parallel_download_failure.json", {"job": job, "error": traceback.format_exc(),\n                        "completed_jobs": reports, "training_started": False})\n                    raise\n                reports[job["key"]] = report\n                print(f"Ready {len(reports)}/{len(jobs)}: {job[\'key\']} ({report[\'seconds\']:.1f}s, reused={report[\'prefix_cache_reused\']})", flush=True)\n            with status_lock:\n                active = [(key, dict(value)) for key, value in statuses.items() if value["stage"] in ("opening stream", "reading")]\n            if pending:\n                details = "; ".join(f"{key}: {value[\'stage\']}, {value[\'rows\']:,} rows, {value[\'candidate_bytes\']/1024**2:.1f} MiB"\n                                    for key, value in active)\n                print(f"Sources ready {len(reports)}/{len(jobs)}; active {len(active)}/{DOWNLOAD_WORKERS}. {details}", flush=True)\n    finally:\n        stop.set()\n        for future in pending:\n            future.cancel()\n        pool.shutdown(wait=True, cancel_futures=True)\n    fetch_seconds = time.monotonic() - started\n\n    # Merge in canonical source order so network completion cannot change the corpus.\n    rules = recipe()\n    seen_hashes, forbidden_groups = set(), set()\n    docs = {"train": [], "in_mix": [], "wiki": []}\n    counts, rejected, summaries = Counter(), Counter(), {}\n    refill_stop = threading.Event()\n    for job in jobs:\n        targets = job["targets"]\n        actual = {key: 0 for key in targets}\n        consumed_parts = 0\n        factor = PREFIX_FACTOR\n        directory = Path(reports[job["key"]]["directory"])\n        while True:\n            state = read_json(directory / "state.json")\n            for part in state["parts"][consumed_parts:]:\n                with (directory / part["name"]).open(encoding="utf-8") as stream:\n                    for line in stream:\n                        candidate = json.loads(line)\n                        split, bucket = candidate["split"], candidate["bucket"]\n                        stratum = split + ":" + bucket\n                        remaining = targets[stratum] - actual[stratum]\n                        if remaining <= rules["byte_quota_tolerance"]:\n                            continue\n                        if split == "train" and candidate["source_group"] in forbidden_groups:\n                            rejected["reserved_evaluation_group"] += 1\n                            continue\n                        span = utf8_prefix(candidate["text"], remaining)\n                        if not span:\n                            continue\n                        digest = hashlib.sha256(span.encode("utf-8")).hexdigest()\n                        full_digest = candidate["original_document_sha256"]\n                        if full_digest in seen_hashes or digest in seen_hashes:\n                            rejected["exact_duplicate"] += 1\n                            continue\n                        seen_hashes.update((full_digest, digest))\n                        if split != "train":\n                            forbidden_groups.add(candidate["source_group"])\n                        length = len(span.encode("utf-8"))\n                        docs[split].append({"text": span, "sha256": digest, "original_document_sha256": full_digest,\n                            "source_group": candidate["source_group"], "source": candidate["source"],\n                            "language": candidate["language"], "domain": candidate["domain"], "bucket": bucket,\n                            "utf8_bytes": length, "source_byte_start": 0, "source_byte_end": length,\n                            "clipped": candidate["source_was_clipped"] or span != candidate["text"]})\n                        actual[stratum] += length\n            consumed_parts = len(state["parts"])\n            if all(targets[key] - actual[key] <= rules["byte_quota_tolerance"] for key in targets):\n                break\n            if state["exhausted"] or state["rows_read"] >= job["row_limit"]:\n                raise RuntimeError(f"Data quota shortfall after deduplication: {job[\'key\']}: {actual}/{targets}. No substitution.")\n            # Continue the same pinned stream when deduplication consumes the buffer.\n            # This is additional data preparation, never another model experiment.\n            factor *= 2\n            print(f"Extending cached prefix after deduplication: {job[\'key\']}", flush=True)\n            prior_rows = state["rows_read"]\n            report = materialize_prefix(job, cache_root, factor, status, refill_stop)\n            reports[job["key"]]["refilled_prefix"] = report\n            if report["rows_read"] == prior_rows:\n                raise RuntimeError("No additional source rows available: " + job["key"])\n        if job["kind"] == "wiki":\n            summaries[job["key"]] = {"bytes": actual["wiki:" + job["key"]], "subset": job["subset"]}\n        else:\n            for bucket in job["buckets"]:\n                key = bucket["key"]\n                summaries[key] = {"targets": {split: targets[split + ":" + key] for split in ("train", "in_mix")},\n                    "actual": {split: actual[split + ":" + key] for split in ("train", "in_mix")},\n                    "repository": job["repo"], "subset": job["subset"]}\n        print("Selected byte quotas: " + job["key"] + " " + json.dumps(actual), flush=True)\n    by_bucket = defaultdict(list)\n    for doc in docs["train"]:\n        by_bucket[doc["bucket"]].append(doc)\n    rng = np.random.default_rng(42)\n    weights = {bucket["key"]: bucket["bp"] / 10000 for bucket in rules["buckets"]}\n    for rows in by_bucket.values():\n        rng.shuffle(rows)\n    used, interleaved = Counter(), []\n    while any(by_bucket.values()):\n        key = min((key for key, rows in by_bucket.items() if rows), key=lambda key: used[key] / weights[key])\n        doc = by_bucket[key].pop()\n        interleaved.append(doc)\n        used[key] += doc["utf8_bytes"]\n    docs["train"] = interleaved\n    train_groups = {doc["source_group"] for doc in docs["train"]}\n    eval_groups = {doc["source_group"] for split in ("in_mix", "wiki") for doc in docs[split]}\n    if train_groups & eval_groups:\n        raise ValueError("Source group overlaps training and evaluation.")\n    for split, rows in docs.items():\n        temporary = destination / (split + ".jsonl.partial")\n        with temporary.open("w", encoding="utf-8") as stream:\n            for doc in rows:\n                stream.write(json.dumps(doc, ensure_ascii=False) + "\\n")\n        temporary.replace(destination / (split + ".jsonl"))\n        counts[split] = len(rows)\n    # Preserve the V3 manifest schema so completed V3 raw/token caches remain reusable.\n    manifest = {"protocol": PROTOCOL, "recipe_sha256": protocol_fingerprint(), "source_revisions": lock,\n        "recipe": rules, "buckets": summaries, "rejected": dict(rejected), "document_counts": dict(counts),\n        "original_bytes": {split: sum(doc["utf8_bytes"] for doc in rows) for split, rows in docs.items()},\n        "language_bytes": {split: {lang: sum(doc["utf8_bytes"] for doc in rows if doc["language"] == lang) for lang in LANGUAGES}\n                           for split, rows in docs.items()},\n        "cross_split_source_groups": 0,\n        "deduplication_scope": "global exact source/span hashes and source groups; near duplicates are not exhaustively removed",\n        "files": {split + ".jsonl": sha256_file(destination / (split + ".jsonl")) for split in docs}}\n    write_json(shared / "parallel_downloads.json", {"version": "V3.6", "max_workers": DOWNLOAD_WORKERS,\n        "source_jobs": len(jobs), "peak_active_source_jobs": peak_active,\n        "fetch_seconds": fetch_seconds, "prepare_raw_seconds": time.monotonic() - started,\n        "jobs": reports, "merge_order": [job["key"] for job in jobs],\n        "initial_prefix_factor": PREFIX_FACTOR, "minimum_candidate_documents_per_stratum": 4,\n        "max_cached_bytes_per_job": PREFIX_JOB_LIMIT_BYTES, "persistent_batch_target_bytes": PREFIX_PART_BYTES,\n        "selection": "canonical V3 source order; independent of worker completion order",\n        "throughput_speedup_measured": False})\n    # The final manifest is the completion marker; it is committed last.\n    write_json(destination / "manifest.json", manifest)\n    return manifest\n\n\ndef prepare_tokenizer(config, destination):\n    from tokenizers import Tokenizer\n    from huggingface_hub import HfApi, snapshot_download\n    destination.mkdir(parents=True, exist_ok=True)\n    reference = config["tokenizer_reference"]\n    if Path(reference).is_dir():\n        for name in ("tokenizer.json", "tokenizer_config.json", "config.json", "special_tokens_map.json"):\n            if (Path(reference) / name).exists():\n                shutil.copyfile(Path(reference) / name, destination / name)\n        revision = None\n    else:\n        revision = HfApi().model_info(reference, revision=config["tokenizer_revision"]).sha\n        snapshot_download(reference, revision=revision, local_dir=destination,\n                          allow_patterns=["tokenizer.json", "tokenizer_config.json", "config.json", "special_tokens_map.json"])\n    tokenizer = Tokenizer.from_file(str(destination / "tokenizer.json"))\n    ids = sorted(set(tokenizer.get_vocab(with_added_tokens=True).values()))\n    # Every tokenizer ID is mapped to a positive dense model ID; model padding is always 0.\n    mapping = {old: index + 1 for index, old in enumerate(ids)}\n    rows = len(ids) + 3\n    total = 100326016 + rows * 640\n    if total >= 300000000:\n        raise ValueError(f"Tokenizer requires {rows} rows and {total:,} parameters, exceeding the strict <300M class.")\n    info = {"reference": reference, "revision": revision, "tokenizer_sha256": sha256_file(destination / "tokenizer.json"),\n        "vocabulary_ids": len(ids), "original_id_space": max(ids) + 1, "model_rows": rows,\n        "pad_id": 0, "bos_id": len(ids) + 1, "eos_id": len(ids) + 2,\n        "id_mapping": "sorted native IDs map to 1..N; PAD=0, BOS=N+1, EOS=N+2",\n        "expected_parameters": total, "model_type": json.loads((destination / "tokenizer.json").read_text())["model"]["type"]}\n    write_json(destination / "identity.json", info)\n    write_json(destination / "native_ids_in_dense_order.json", ids)\n    return tokenizer, mapping, info\n\n\ndef tokenize_split(documents, tokenizer, mapping, identity, sequence_length):\n    arrays = defaultdict(list)\n    text_token_count = 0\n    for doc in documents:\n        encoded = tokenizer.encode(doc["text"], add_special_tokens=False)\n        if tokenizer.decode(encoded.ids, skip_special_tokens=False).encode("utf-8") != doc["text"].encode("utf-8"):\n            raise ValueError(f"Tokenizer is not byte-lossless on fixed input {doc[\'sha256\']} ({doc[\'language\']}); corpus was not modified.")\n        ids = [mapping[index] for index in encoded.ids]\n        if not ids:\n            raise ValueError("Nonempty source text produced zero tokens.")\n        # Attribute each original byte once using the monotonic original-character offset frontier.\n        char_bytes = np.concatenate(([0], np.cumsum([len(c.encode("utf-8")) for c in doc["text"]])))\n        frontier = np.maximum.accumulate(np.array([0] + [end for _, end in encoded.offsets], np.int64))\n        frontier = np.clip(frontier, 0, len(doc["text"]))\n        frontier[-1] = len(doc["text"])\n        byte_weights = np.diff(char_bytes[frontier]).astype(np.int32)\n        sequence = [identity["bos_id"], *ids, identity["eos_id"]]\n        weights = np.concatenate((byte_weights, [0])).astype(np.int32)\n        for start in range(0, len(sequence) - 1, sequence_length):\n            window = sequence[start:start + sequence_length + 1]\n            size = len(window) - 1\n            for key, values, dtype, padding in (\n                ("tokens", window[:-1], np.int32, 0), ("targets", window[1:], np.int32, 0),\n                ("mask", [True] * size, np.bool_, False),\n                ("text_mask", [start + k < len(ids) for k in range(size)], np.bool_, False),\n                ("byte_weights", weights[start:start + size], np.int32, 0),\n                ("language_ids", [LANGUAGES.index(doc["language"])] * size, np.int32, 0),\n                ("domain_ids", [DOMAINS.index(doc["domain"])] * size, np.int32, 0)):\n                row = np.full(sequence_length, padding, dtype)\n                row[:size] = values\n                arrays[key].append(row)\n        text_token_count += len(ids)\n    result = {key: np.stack(value) for key, value in arrays.items()}\n    if int(result["byte_weights"].sum()) != sum(doc["utf8_bytes"] for doc in documents):\n        raise ValueError("Original-byte attribution failed.")\n    packed, packing = pack_chunks(result, 42)\n    for key in EXTRA_KEYS:\n        packed[key] = np.zeros_like(packed["tokens"], dtype=result[key].dtype)\n        for source, target, start, length in packing["assignments_source_bin_start_length"]:\n            packed[key][target, start:start + length] = result[key][source, :length]\n    packing["text_tokens"] = text_token_count\n    packing["original_bytes"] = int(packed["byte_weights"].sum())\n    return packed, packing\n\n\ndef prepare_cached(config, work, output):\n    started = time.monotonic()\n    work, output = Path(work), Path(output)\n    shared = output.parent.parent / "V3-shared" / protocol_fingerprint()[:16]\n    shared.mkdir(parents=True, exist_ok=True)\n    lock = immutable_sources(shared)\n    raw = shared / "raw"\n    if not raw.exists():\n        staging = shared / "raw.partial"\n        build_raw_corpus(staging, lock)\n        staging.rename(raw)\n        raw_reused = bool(read_json(shared / "parallel_downloads.json").get("recovered_complete_staging", False))\n    else:\n        raw_reused = True\n    raw_manifest = verify_raw(raw)\n    prepared = work / "prepared"\n    prepared.mkdir(exist_ok=False)\n    download_report_path = shared / "parallel_downloads.json"\n    if raw_reused:\n        download_report = {"version": "V3.6", "raw_cache_reused": True, "source_jobs": 0,\n            "max_workers": DOWNLOAD_WORKERS, "peak_active_source_jobs": 0, "fetch_seconds": 0.,\n            "note": "Verified completed raw corpus reused; no source fetch was needed."}\n    else:\n        download_report = read_json(download_report_path)\n        download_report["raw_cache_reused"] = False\n    if download_report["peak_active_source_jobs"] > DOWNLOAD_WORKERS:\n        raise ValueError("Source download concurrency exceeded six jobs.")\n    write_json(output / "parallel_downloads.json", download_report)\n    preparation_checks = {\n        "V36-021_DOWNLOAD_BOUND": download_report["peak_active_source_jobs"] <= 6,\n        "V36-022_PREFIX_CACHE": raw_reused or all(Path(item["directory"]).joinpath("state.json").exists()\n                                                    for item in download_report.get("jobs", {}).values()),\n        "V36-023_MERGE_ORDER": raw_reused or download_report.get("merge_order", [])[:12] == ["wiki_" + lang for lang in LANGUAGES],\n        "V36-024_SOURCE_COMPLETE": raw_reused or download_report.get("source_jobs") == 30 and len(download_report.get("jobs", {})) == 30,\n    }\n    write_json(output / "preparation_requirements.json", {key: "PASS" if passed else "FAIL" for key, passed in preparation_checks.items()})\n    if not all(preparation_checks.values()):\n        raise ValueError("V3.6 preparation engineering requirement failed.")\n    tokenizer, mapping, identity = prepare_tokenizer(config, prepared / "tokenizer")\n    token_key = hashlib.sha256((identity["tokenizer_sha256"] + sha256_file(raw / "manifest.json") + PACKING_VERSION).encode()).hexdigest()[:24]\n    token_cache = shared / "tokenized" / token_key\n    if token_cache.exists():\n        cached = read_json(token_cache / "tokenization.json")\n        for name, digest in cached["files"].items():\n            if sha256_file(token_cache / name) != digest:\n                raise ValueError("Cached tokenization checksum mismatch.")\n            shutil.copyfile(token_cache / name, prepared / name)\n        shutil.copyfile(token_cache / "tokenization.json", prepared / "tokenization.json")\n        token_reused = True\n    else:\n        details = {"tokenizer": identity, "raw_manifest_sha256": sha256_file(raw / "manifest.json"), "splits": {}, "files": {}}\n        for split in ("train", "in_mix", "wiki"):\n            arrays, packing = tokenize_split(read_jsonl(raw / (split + ".jsonl")), tokenizer, mapping, identity, config["sequence_length"])\n            name = split + ".npz"\n            np.savez_compressed(prepared / name, **arrays)\n            details["splits"][split] = {key: value for key, value in packing.items() if not key.startswith("assignments_")}\n            details["files"][name] = sha256_file(prepared / name)\n            print(f"Tokenized {split}: {packing[\'text_tokens\']:,} text targets; occupancy={packing[\'occupancy\']:.2%}", flush=True)\n        write_json(prepared / "tokenization.json", details)\n        token_cache.parent.mkdir(parents=True, exist_ok=True)\n        token_cache.mkdir(exist_ok=False)\n        for name in [*details["files"], "tokenization.json"]:\n            shutil.copyfile(prepared / name, token_cache / name)\n        token_reused = False\n    config = {**config, "tokenizer_identity": identity, "shared_directory": str(shared)}\n    write_json(prepared / "resolved_config.json", config)\n    shutil.copyfile(raw / "manifest.json", prepared / "manifest.json")\n    for source, destination in ((prepared / "manifest.json", output / "data_manifest.json"),\n                                (prepared / "tokenization.json", output / "tokenization.json"),\n                                (prepared / "tokenizer/identity.json", output / "tokenizer_identity.json"),\n                                (shared / "source-lock.json", output / "source-lock.json")):\n        shutil.copyfile(source, destination)\n    shutil.copytree(prepared / "tokenizer", output / "tokenizer")\n    write_json(output / "preparation.json", {"raw_cache_reused": raw_reused, "token_cache_reused": token_reused,\n        "preparation_seconds": time.monotonic() - started, "raw_manifest_sha256": sha256_file(raw / "manifest.json"),\n        "tokenizer_sha256": identity["tokenizer_sha256"], "shared_directory": str(shared)})\n\n\ndef load_model_helpers(cpu_test=False):\n    accelerator_imports(cpu_test)\n    exec(MODEL_HELPERS, globals())\n\n\ndef make_model(config):\n    from gemma import gm\n    model = gm.nn.Gemma3_270M(dtype=jnp.float32)\n    return model.clone(config=dataclasses.replace(model.config, num_embed=config["tokenizer_identity"]["model_rows"]))\n\n\nPILOT_VERSION = \'V3.6\'\n\ndef make_optimizer(config, params):\n    """All-parameter Muon, including a declared one-row extension for vectors.\n\n    Use the public low-level Muon transform: the high-level muon() wrapper has\n    an AdamW branch. No AdamW transform is constructed here. RMS shape scaling\n    is explicit, so the very wide tied table and one-row normalization vectors\n    do not inherit unrelated width-dependent update magnitudes.\n    """\n    flat = traverse_util.flatten_dict(params)\n    specs, scales, routing = {}, {}, []\n    for path, value in sorted(flat.items()):\n        name = "/".join(path)\n        shape = tuple(value.shape)\n        lifted_shape = (1, value.size) if value.ndim < 2 else shape\n        if value.ndim < 2:\n            reduction, output = (0,), (1,)\n        elif path == ("embedder", "input_embedding"):\n            # Native decoding multiplies hidden states by this table\'s transpose.\n            reduction, output = (1,), (0,)\n        elif "q_einsum" in name:\n            reduction, output = (1,), (0, 2)\n        elif "kv_einsum" in name:\n            reduction, output = (2,), (1, 3)\n        elif "attn_vec_einsum" in name:\n            reduction, output = (0, 1), (2,)\n        elif path[-1] == "gating_einsum":\n            reduction, output = (2,), (1,)\n        elif value.ndim == 2:\n            reduction, output = (0,), (1,)\n        else:\n            raise ValueError(f"Unclassified all-Muon parameter: {name} {shape}")\n        specs[path] = optax.contrib.MuonDimensionNumbers(\n            reduction_axis=reduction, output_axis=output)\n        fan_in = math.prod(lifted_shape[axis] for axis in reduction)\n        fan_out = math.prod(lifted_shape[axis] for axis in output)\n        scales[path] = config["muon_consistent_rms"] * math.sqrt(max(fan_in, fan_out))\n        routing.append({"path": name, "shape": list(shape),\n            "lifted_shape": list(lifted_shape), "parameters": value.size,\n            "optimizer": "Muon", "reduction_axes": list(reduction),\n            "output_axes": list(output), "shape_scale": scales[path],\n            "vector_extension": value.ndim < 2, "weight_decay": value.ndim > 1})\n    spec_tree = traverse_util.unflatten_dict(specs)\n    scale_tree = traverse_util.unflatten_dict(scales)\n    decay = jax.tree.map(lambda value: value.ndim > 1, params)\n    def lift(tree):\n        return jax.tree.map(lambda value: value.reshape((1, value.size))\n                            if value.ndim < 2 else value, tree)\n    def shape_update(updates, state, params=None):\n        del params\n        return jax.tree.map(lambda value, scale: value * scale, updates, scale_tree), state\n    schedule = optax.join_schedules([\n        optax.linear_schedule(0., config["learning_rate"], config["warmup_updates"]),\n        optax.constant_schedule(config["learning_rate"]),\n    ], [config["warmup_updates"]])\n    inner = optax.chain(\n        optax.clip_by_global_norm(config["gradient_clip"]),\n        optax.contrib.scale_by_muon(ns_steps=config["muon_ns_steps"],\n            beta=config["muon_beta"], eps=1e-8, mu_dtype=jnp.float32,\n            nesterov=True, weight_dimension_numbers=spec_tree),\n        optax.GradientTransformation(lambda _: optax.EmptyState(), shape_update),\n        optax.add_decayed_weights(config["weight_decay"], mask=decay),\n        optax.scale_by_learning_rate(schedule))\n    def init_fn(parameters):\n        return inner.init(lift(parameters))\n    def update_fn(gradients, state, parameters=None):\n        if parameters is None:\n            raise ValueError("All-Muon requires parameters for decoupled weight decay.")\n        updates, new_state = inner.update(lift(gradients), state, lift(parameters))\n        updates = jax.tree.map(lambda update, parameter: update.reshape(parameter.shape),\n                               updates, parameters)\n        return updates, new_state\n    count = sum(value.size for value in flat.values())\n    vector_count = sum(value.size for value in flat.values() if value.ndim < 2)\n    return optax.GradientTransformation(init_fn, update_fn), {\n        "name": "Muon-all-with-vector-extension", "muon_parameters": count,\n        "adamw_parameters": 0, "learning_rate": config["learning_rate"],\n        "beta": config["muon_beta"], "newton_schulz_steps": config["muon_ns_steps"],\n        "nesterov": True, "momentum_dtype": "float32",\n        "shape_scaling": "consistent RMS", "consistent_rms": config["muon_consistent_rms"],\n        "nominal_lr_times_rms": config["learning_rate"] * config["muon_consistent_rms"],\n        "embedding_parameters": flat[("embedder", "input_embedding")].size,\n        "vector_extension_parameters": vector_count,\n        "vector_extension": "rank-0/1 parameters reshaped to 1 x N for Muon, then restored",\n        "warmup_updates": config["warmup_updates"], "routing": routing}\n\n\ndef readable_evaluation(values, raw_bytes):\n    ln, lb, dn, db, eos = [np.asarray(value) for value in values]\n    if int(round(float(lb.sum()))) != raw_bytes or not np.isfinite(ln).all():\n        raise ValueError("Evaluation original-byte denominator or finite-loss check failed.")\n    languages = {language: {"bpb": float(ln[i] / (lb[i] * math.log(2))), "nll_nats": float(ln[i]),\n                            "original_utf8_bytes": int(lb[i])} for i, language in enumerate(LANGUAGES) if lb[i] > 0}\n    domains = {domain: {"bpb": float(dn[i] / (db[i] * math.log(2))), "original_utf8_bytes": int(db[i])}\n               for i, domain in enumerate(DOMAINS) if db[i] > 0}\n    return {"pooled_bpb": float(ln.sum() / (lb.sum() * math.log(2))),\n        "language_macro_bpb": float(np.mean([value["bpb"] for value in languages.values()])),\n        "languages": languages, "domains": domains, "original_utf8_bytes": raw_bytes,\n        "eos_nll_nats_separate": float(eos), "scoring": "all original text targets once; virtual EOS excluded from BPB; fixed 1024-token chunk context"}\n\n\ndef initial_backbone_digest(params):\n    digest = hashlib.sha256()\n    for path, value in sorted(traverse_util.flatten_dict(params).items()):\n        if path[0] == "embedder":\n            continue\n        digest.update("/".join(path).encode())\n        digest.update(np.asarray(jax.device_get(value)).tobytes())\n    return digest.hexdigest()\n\n\n\n"""Repair the Tokamax loss gradient boundary without changing its TPU math."""\n\n\n\n\n# Shared native-Gemma runtime assembled into both new notebooks.\n# This source is not executed by the artifact builder.\n\nTRAIN_KEYS = ("tokens", "targets", "mask", "positions", "segment_ids")\n\n\ndef token_losses(model, params, batch):\n    compute = jax.tree.map(lambda value: value.astype(jnp.bfloat16), params)\n    logits = model.apply({"params": compute}, tokens=batch["tokens"],\n        positions=batch["positions"],\n        attention_mask=packed_attention_mask(batch["segment_ids"])).logits.astype(jnp.float32)\n    selected = jnp.take_along_axis(logits, batch["targets"][..., None], axis=-1)[..., 0]\n    return jax.scipy.special.logsumexp(logits, axis=-1) - selected\n\n\ndef make_training_functions(model, optimizer):\n    def one_step(state, batch):\n        def objective(params):\n            losses = token_losses(model, params, batch)\n            count = jnp.sum(batch["mask"], dtype=jnp.float32)\n            nll = jnp.sum(jnp.where(batch["mask"], losses, 0.))\n            return nll / jnp.maximum(count, 1.), count\n        (loss, count), grads = jax.value_and_grad(objective, has_aux=True)(state.params)\n        norm = optax.global_norm(grads)\n        finite = jnp.isfinite(loss) & jnp.isfinite(norm) & (count > 0)\n        def apply(_):\n            delta, new_opt = optimizer.update(grads, state.opt_state, state.params)\n            return state.replace(step=state.step + 1,\n                params=optax.apply_updates(state.params, delta), opt_state=new_opt,\n                data_position=state.data_position + 1)\n        state = jax.lax.cond(finite, apply, lambda _: state, operand=None)\n        return state, (loss, count, norm, finite)\n    def group(state, training, requested):\n        initial = (state, jnp.array(0, jnp.int32), jnp.array(0., jnp.float32),\n                   jnp.array(0, jnp.int32), jnp.array(0., jnp.float32), jnp.array(True))\n        def condition(carry):\n            return (carry[1] < requested) & carry[5]\n        def body(carry):\n            state, steps, nll, count, max_norm, finite = carry\n            index = state.data_position % training["tokens"].shape[0]\n            batch = jax.tree.map(lambda values: jax.lax.dynamic_index_in_dim(\n                values, index, keepdims=False), training)\n            state, (loss, tokens, norm, good) = one_step(state, batch)\n            return (state, steps + good.astype(jnp.int32), nll + loss * tokens,\n                count + jnp.where(good, tokens, 0).astype(jnp.int32),\n                jnp.maximum(max_norm, norm), finite & good)\n        state, steps, nll, count, norm, finite = jax.lax.while_loop(condition, body, initial)\n        return state, {"updates": steps, "nll_sum": nll, "tokens": count,\n                       "max_gradient_norm": norm, "finite": finite}\n    return one_step, jax.jit(group, donate_argnums=(0,))\n\n\ndef make_evaluator(model):\n    """One executable for both held-out sets; split identity is runtime data."""\n    def evaluate(params, arrays):\n        initial = (jnp.zeros((2, len(LANGUAGES)), jnp.float32),\n                   jnp.zeros((2, len(LANGUAGES)), jnp.float32),\n                   jnp.zeros((2, len(DOMAINS)), jnp.float32),\n                   jnp.zeros((2, len(DOMAINS)), jnp.float32), jnp.zeros(2, jnp.float32))\n        def body(index, totals):\n            batch = jax.tree.map(lambda values: jax.lax.dynamic_index_in_dim(\n                values, index, keepdims=False), arrays)\n            losses = token_losses(model, params, batch)\n            nll = jnp.where(batch["text_mask"], losses, 0.).reshape(-1)\n            weights = batch["byte_weights"].reshape(-1).astype(jnp.float32)\n            language = batch["language_ids"].reshape(-1) + batch["split_id"] * len(LANGUAGES)\n            domain = batch["domain_ids"].reshape(-1) + batch["split_id"] * len(DOMAINS)\n            ln = jnp.bincount(language, weights=nll, length=2 * len(LANGUAGES)).reshape((2, -1))\n            lb = jnp.bincount(language, weights=weights, length=2 * len(LANGUAGES)).reshape((2, -1))\n            dn = jnp.bincount(domain, weights=nll, length=2 * len(DOMAINS)).reshape((2, -1))\n            db = jnp.bincount(domain, weights=weights, length=2 * len(DOMAINS)).reshape((2, -1))\n            eos = jnp.sum(jnp.where(batch["mask"] & ~batch["text_mask"], losses, 0.))\n            eos = jax.nn.one_hot(batch["split_id"], 2) * eos\n            return tuple(total + value for total, value in zip(totals, (ln, lb, dn, db, eos)))\n        return jax.lax.fori_loop(0, arrays["tokens"].shape[0], body, initial)\n    return jax.jit(evaluate)\n\n\ndef training_prefix_counts(arrays):\n    """CPU preparation only; no per-token source histogram in the training graph."""\n    batches = arrays["tokens"].shape[0]\n    counts = {"tokens": arrays["mask"].sum(axis=(1, 2), dtype=np.int64)}\n    weights = arrays["byte_weights"].reshape(-1)\n    for key, field, classes in (("domain_bytes", "domain_ids", len(DOMAINS)),\n                                 ("language_bytes", "language_ids", len(LANGUAGES))):\n        labels = arrays[field].reshape((batches, -1))\n        indices = (labels + np.arange(batches, dtype=np.int64)[:, None] * classes).reshape(-1)\n        counts[key] = np.bincount(indices, weights=weights,\n            minlength=batches * classes).astype(np.int64).reshape((batches, classes))\n    if int(counts["domain_bytes"].sum()) != int(counts["language_bytes"].sum()):\n        raise ValueError("Prepared source/language byte accounting differs.")\n    return {key: np.concatenate((np.zeros((1, *value.shape[1:]), np.int64),\n                np.cumsum(value, axis=0, dtype=np.int64)), axis=0)\n            for key, value in counts.items()}\n\n\ndef completed_prefix(prefix, updates):\n    batches = prefix.shape[0] - 1\n    passes, remainder = divmod(updates, batches)\n    return passes * prefix[-1] + prefix[remainder]\n\n\ndef verify_native_protocol():\n    started = time.monotonic()\n    segments = jnp.array([[0, 0, 1, 1, -1]], jnp.int32)\n    actual = np.asarray(jax.jit(packed_attention_mask)(segments))[0]\n    expected = np.array([[1, 0, 0, 0, 0], [1, 1, 0, 0, 0], [0, 0, 1, 0, 0],\n                         [0, 0, 1, 1, 0], [0, 0, 0, 0, 1]], np.bool_)\n    passed = bool(np.array_equal(actual, expected))\n    if not passed:\n        raise ValueError("Native packed attention mask failed causal/segment/padding verification.")\n    return {"passed": passed, "seconds": time.monotonic() - started,\n        "optimizer_updates": 0, "attention": "native Gemma", "loss": "native dense exact softmax",\n        "custom_kernel_equivalence": "not applicable; no Splash or Tokamax execution",\n        "mask_verified": passed}\n\n\ndef train_pilot(work, output):\n    work, output = Path(work), Path(output)\n    version = PILOT_VERSION\n    with (output / "training.started").open("x") as marker:\n        marker.write(version + ": one training attempt; automatic retry disabled.\\n")\n    write_json(output / "attempt_status.json", {"stage": "pretraining", "optimizer_updates": 0,\n        "timed_training_started": False, "version": version})\n    environment = preflight()\n    write_json(output / "preflight.json", environment)\n    load_model_helpers()\n    data_dir = work / "prepared"\n    config = read_json(data_dir / "resolved_config.json")\n    manifest = verify_raw(Path(config["shared_directory"]) / "raw")\n    tokenization = read_json(data_dir / "tokenization.json")\n    identity = config["tokenizer_identity"]\n    if tokenization["raw_manifest_sha256"] != sha256_file(data_dir / "manifest.json"):\n        raise ValueError("Tokenized and raw corpus manifests differ.")\n    if shutil.disk_usage(work).free < 3 * 1024**3:\n        raise RuntimeError("The weights-only artifact requires at least 3 GiB free local disk.")\n    host, orders = {}, {}\n    for split in ("train", "in_mix", "wiki"):\n        name = split + ".npz"\n        if sha256_file(data_dir / name) != tokenization["files"][name]:\n            raise ValueError("Prepared input checksum failed: " + name)\n        with np.load(data_dir / name, allow_pickle=False) as archive:\n            arrays = {key: archive[key] for key in INPUT_KEYS}\n        if int(arrays["byte_weights"].sum()) != manifest["original_bytes"][split]:\n            raise ValueError("Prepared original-byte count differs: " + split)\n        host[split], orders[split] = batch_training_arrays(\n            arrays, config["local_batch_per_chip"], config["seed"])\n    phase = time.monotonic()\n    prefix = training_prefix_counts(host["train"])\n    write_json(output / "training_accounting.json", {\n        "method": "CPU batch prefix sums; completed update cursor; no timed per-token histograms",\n        "training_batches": host["train"]["tokens"].shape[0],\n        "tokens_per_pass": int(prefix["tokens"][-1]),\n        "domain_bytes_per_pass": prefix["domain_bytes"][-1].tolist(),\n        "language_bytes_per_pass": prefix["language_bytes"][-1].tolist(),\n        "seconds": time.monotonic() - phase})\n    training_host = {key: host["train"][key] for key in TRAIN_KEYS}\n    eval_host = {key: np.concatenate((host["in_mix"][key], host["wiki"][key]), axis=0)\n                 for key in INPUT_KEYS}\n    eval_host["split_id"] = np.concatenate((\n        np.zeros(host["in_mix"]["tokens"].shape[0], np.int32),\n        np.ones(host["wiki"]["tokens"].shape[0], np.int32)))\n    input_bytes = sum(value.nbytes for value in training_host.values())\n    phase = time.monotonic()\n    training, validation = jax.device_put((training_host, eval_host))\n    jax.block_until_ready((training, validation))\n    input_transfer_seconds = time.monotonic() - phase\n    resident = all(jax.devices()[0] in leaf.devices() for leaf in jax.tree.leaves(training))\n    del host, arrays, training_host, eval_host\n    print(f"Training inputs on TPU: {input_bytes / 1024**2:.1f} MiB; CPU byte accounting prepared.", flush=True)\n    write_json(output / "config.json", config)\n    write_json(output / "environment.json", environment)\n\n    phase = time.monotonic()\n    model = make_model(config)\n    if model.config.final_logit_softcap is not None:\n        raise ValueError("This protocol requires native uncapped Gemma 270M logits.")\n    params = jax.jit(lambda key: model.init(key, tokens=jnp.ones((1, 1), jnp.int32))["params"])(\n        jax.random.PRNGKey(config["seed"]))\n    jax.block_until_ready(params)\n    flat = traverse_util.flatten_dict(params)\n    parameter_count = sum(value.size for value in flat.values())\n    embedding_shape = (identity["model_rows"], 640)\n    tied = (flat[("embedder", "input_embedding")].shape == embedding_shape\n            and sum(value.shape == embedding_shape for value in flat.values()) == 1)\n    if parameter_count != identity["expected_parameters"] or not tied or not all(\n            value.dtype == jnp.float32 for value in flat.values()):\n        raise ValueError("Parameter count, tied table, or FP32 master weights differ.")\n    backbone_digest = initial_backbone_digest(params)\n    backbone_key = hashlib.sha256(json.dumps({"gemma": environment["packages"].get("gemma"),\n        "jax": environment["jax"], "seed": config["seed"], "backbone": "Gemma3_270M"},\n        sort_keys=True).encode()).hexdigest()[:16]\n    fingerprint_path = Path(config["shared_directory"]) / ("backbone-" + backbone_key + ".json")\n    fingerprint = {"sha256": backbone_digest, "transformer_and_norm_parameters": 100326016,\n                   "seed": config["seed"]}\n    if fingerprint_path.exists() and read_json(fingerprint_path) != fingerprint:\n        raise ValueError("Transformer initialization differs from the shared baseline.")\n    write_json(fingerprint_path, fingerprint)\n    write_json(output / "backbone_initialization.json", fingerprint)\n    probe_path = next(path for path, value in sorted(flat.items())\n                      if path[0].startswith("layer_") and value.ndim > 1)\n    def probe(parameters):\n        return np.asarray(jax.device_get(traverse_util.flatten_dict(parameters)[probe_path].reshape(-1)[:64]))\n    initial_probe = probe(params)\n    optimizer, partition = make_optimizer(config, params)\n    routing = partition.pop("routing")\n    routing_ok = (sum(row["parameters"] for row in routing) == parameter_count\n                  and all(row["optimizer"] == config["optimizer_label"] for row in routing)\n                  and partition["muon_parameters"] + partition["adamw_parameters"] == parameter_count)\n    if not routing_ok:\n        raise ValueError("Optimizer routing does not cover every parameter with the requested optimizer.")\n    write_json(output / "optimizer_parameter_routing.json", {"parameters": parameter_count,\n        "optimizer": config["optimizer_label"], "all_parameters_accounted_for": routing_ok,\n        "leaves": routing})\n    write_json(output / "optimizer_partition.json", partition)\n    state = PilotState(jnp.array(0, jnp.int32), params, optimizer.init(params),\n        jax.random.PRNGKey(config["seed"]), jnp.array(0, jnp.int32))\n    del params, flat\n    jax.block_until_ready(state)\n    initialization_seconds = time.monotonic() - phase\n    protocol_report = verify_native_protocol()\n    write_json(output / "kernel_verification.json", protocol_report)\n    _, group_jit = make_training_functions(model, optimizer)\n    evaluation_jit = make_evaluator(model)\n    print("Compiling one native training loop and one combined held-out evaluator; zero updates...", flush=True)\n    phase = time.monotonic()\n    executable = group_jit.lower(state, training, jnp.array(0, jnp.int32)).compile()\n    evaluator = evaluation_jit.lower(state.params, validation).compile()\n    state, warmup_metrics = jax.block_until_ready(group_jit(state, training, jnp.array(0, jnp.int32)))\n    if int(warmup_metrics["updates"]) != 0 or int(state.step) != 0 or int(state.data_position) != 0:\n        raise ValueError("The zero-update JIT preparation changed optimizer/data state.")\n    compilation_seconds = time.monotonic() - phase\n    write_json(output / "dispatch_preparation.json", {"dispatch": "jax.jit wrapper",\n        "training_warmup_requested_updates": 0, "training_warmup_completed_updates": 0,\n        "pretraining_optimizer_step": 0, "pretraining_data_position": 0,\n        "training_executables": 1, "evaluation_executables": 1,\n        "compiled_objects_used_for": "precompilation and memory inspection only"})\n    try:\n        analysis = executable.memory_analysis()\n        compiled_memory = {name: int(getattr(analysis, name)) for name in\n            ("argument_size_in_bytes", "output_size_in_bytes", "alias_size_in_bytes", "temp_size_in_bytes")}\n    except (AttributeError, TypeError, NotImplementedError):\n        compiled_memory = None\n    def evaluate(parameters):\n        values = jax.block_until_ready(evaluation_jit(parameters, validation))\n        return {split: readable_evaluation(tuple(value[index] for value in values),\n                    manifest["original_bytes"][split])\n                for index, split in enumerate(("in_mix", "wiki"))}\n    phase = time.monotonic()\n    before = evaluate(state.params)\n    initial_eval_seconds = time.monotonic() - phase\n    write_json(output / "timed_training.started.json", {"version": version,\n        "stage": "timed training", "training_seconds": config["training_seconds"]})\n    write_json(output / "attempt_status.json", {"version": version, "stage": "timed training",\n        "timed_training_started": True, "optimizer_updates": None})\n    print(f"Initial FineWiki macro BPB={before[\'wiki\'][\'language_macro_bpb\']:.5f}. Starting ONE 60-second {version} run...", flush=True)\n    log_rows = []\n    updates = tokens_processed = 0\n    max_seconds_per_update = last_progress = last_start = 0.\n    jax.block_until_ready(state)\n    # Preserve completed dispatches even if a later call fails. Buffered log writes\n    # contain scalar metrics only; source/language accounting runs after timing.\n    with (output / "training.jsonl").open("x") as stream:\n        started = time.monotonic()\n        try:\n            while True:\n                remaining = config["training_seconds"] - (time.monotonic() - started)\n                if remaining <= 0:\n                    break\n                group_size = config["updates_per_dispatch"]\n                requested = group_size if updates > 0 and remaining > max(\n                    2., 2 * group_size * max_seconds_per_update) else 1\n                last_start = time.monotonic() - started\n                if last_start >= config["training_seconds"]:\n                    break\n                call_start = time.monotonic()\n                state, values = jax.block_until_ready(group_jit(state, training, jnp.array(requested, jnp.int32)))\n                metrics = {key: np.asarray(value).tolist() for key, value in values.items()}\n                completed = int(metrics["updates"])\n                updates += completed\n                tokens_processed += int(metrics["tokens"])\n                end = time.monotonic() - started\n                row = {**metrics, "update": updates, "requested_updates": requested,\n                    "start_seconds": last_start, "end_seconds": end,\n                    "loss": metrics["nll_sum"] / max(metrics["tokens"], 1)}\n                log_rows.append(row)\n                safe_row = {key: (None if isinstance(value, float) and not math.isfinite(value) else value)\n                            for key, value in row.items()}\n                stream.write(json.dumps(safe_row, allow_nan=False) + "\\n")\n                if not metrics["finite"] or completed != requested:\n                    stream.flush()\n                    raise FloatingPointError("Nonfinite loss/gradient; affected update rejected. No retry.")\n                max_seconds_per_update = max(max_seconds_per_update,\n                    (time.monotonic() - call_start) / completed)\n                if len(log_rows) == 1 or end - last_progress >= 2:\n                    print(f"updates={updates:,} tokens={tokens_processed:,} loss={row[\'loss\']:.5f} elapsed={end:.2f}s", flush=True)\n                    last_progress = end\n            jax.block_until_ready(state)\n            training_seconds = time.monotonic() - started\n        except Exception:\n            stream.flush()\n            write_json(output / "attempt_status.json", {"version": version,\n                "stage": "training failed", "timed_training_started": True,\n                "optimizer_updates": updates, "data_position": int(state.data_position)})\n            raise\n    if not log_rows:\n        raise RuntimeError("No optimizer updates completed.")\n    domain_bytes = completed_prefix(prefix["domain_bytes"], updates)\n    language_bytes = completed_prefix(prefix["language_bytes"], updates)\n    expected_tokens = int(completed_prefix(prefix["tokens"], updates))\n    if tokens_processed != expected_tokens:\n        raise ValueError("Device token count differs from completed CPU batch prefix.")\n    changed = not np.array_equal(initial_probe, probe(state.params))\n    final_finite = bool(jax.device_get(jax.jit(lambda parameters: jnp.all(jnp.stack([\n        jnp.all(jnp.isfinite(value)) for value in jax.tree.leaves(parameters)])))(state.params)))\n    phase = time.monotonic()\n    after = evaluate(state.params)\n    final_eval_seconds = time.monotonic() - phase\n    memory_stats = jax.devices()[0].memory_stats()\n    phase = time.monotonic()\n    metadata = {"version": version, "protocol": PROTOCOL, "config": config,\n        "parameter_count": parameter_count, "initialization": "random; zero pretrained weights",\n        "backbone_initialization_sha256": backbone_digest, "optimizer": partition,\n        "raw_manifest_sha256": sha256_file(data_dir / "manifest.json"),\n        "tokenization_manifest_sha256": sha256_file(data_dir / "tokenization.json"),\n        "packing_version": PACKING_VERSION, "data_order_sha256": orders,\n        "training_batches": training["tokens"].shape[0], "updates": updates,\n        "data_position": int(state.data_position), "checkpoint_fields": ["params"],\n        "checkpoint_kind": "weights_only", "optimizer_resume_supported": False}\n    print("Saving ONE FP32 weights-only artifact; optimizer resume is not included...", flush=True)\n    local_checkpoint = work / "weights.npz"\n    leaves = save_archive(local_checkpoint, {"params": state.params}, metadata)\n    digest = sha256_file(local_checkpoint)\n    partial = output / "weights.npz.partial"\n    shutil.copyfile(local_checkpoint, partial)\n    if sha256_file(partial) != digest:\n        raise ValueError("Drive weights checksum differs from the local artifact.")\n    checkpoint = output / "weights.npz"\n    partial.rename(checkpoint)\n    with zipfile.ZipFile(checkpoint) as archive:\n        saved = json.loads(archive.read("checkpoint.json"))\n        schema_verified = (set(saved["tree"]) == {"params"} and saved["metadata"] == metadata\n            and sum(name.startswith("arrays/") for name in archive.namelist()) == leaves)\n    if not schema_verified:\n        raise ValueError("Saved weights schema or array count differs.")\n    write_json(output / "checkpoint_manifest.json", {"sha256": digest,\n        "bytes": checkpoint.stat().st_size, "array_leaves": leaves,\n        "kind": "weights_only", "optimizer_resume_supported": False,\n        "verification": "local/Drive SHA256 equality plus weights schema and leaf count",\n        "restore_executed": False})\n    checkpoint_seconds = time.monotonic() - phase\n    bytes_processed = int(domain_bytes.sum())\n    prefix_id = config["requirement_prefix"]\n    checks = {\n        "001_ONE_ATTEMPT": (output / "training.started").exists(),\n        "002_ONE_V5E": environment["platform"] == "tpu" and environment["device_count"] == 1,\n        "003_SOURCE_PINS": set(manifest["source_revisions"]) == set(SOURCE_REPOS),\n        "004_FROZEN_CORPUS": tokenization["raw_manifest_sha256"] == sha256_file(data_dir / "manifest.json"),\n        "005_SPLIT_SEPARATION": manifest["cross_split_source_groups"] == 0,\n        "006_LOSSLESS_TOKENIZATION": all(tokenization["splits"][split]["original_bytes"] == manifest["original_bytes"][split]\n                                          for split in ("train", "in_mix", "wiki")),\n        "007_INTACT_PACKING": all(tokenization["splits"][split]["targets_preserved"] and\n            tokenization["splits"][split]["context_preserved"] for split in ("train", "in_mix", "wiki")),\n        "008_NATIVE_ATTENTION_MASK": protocol_report["passed"],\n        "009_RESIDENT_INPUTS": resident,\n        "010_GEMMA_BACKBONE": parameter_count == identity["expected_parameters"] and parameter_count < 300000000 and tied,\n        "011_OPTIMIZER_ROUTING": routing_ok,\n        "012_TIMED_DISPATCH": all(row["start_seconds"] < config["training_seconds"] for row in log_rows),\n        "013_DISPATCH_BOUND": all(1 <= row["updates"] <= 16 for row in log_rows),\n        "014_FINITE_UPDATES": all(row["finite"] for row in log_rows) and final_finite,\n        "015_ZERO_EXTRA_UPDATES": int(state.step) == updates and int(state.data_position) == updates,\n        "016_FIXED_EVALUATION": len(after["wiki"]["languages"]) == 12 and all(\n            after[split]["original_utf8_bytes"] == before[split]["original_utf8_bytes"] for split in ("in_mix", "wiki")),\n        "017_WEIGHTS_ARTIFACT": schema_verified,\n        "018_CHANGED_WEIGHTS": changed,\n        "019_MINIMUM_UPDATES": updates >= config["minimum_updates"],\n        "020_BYTE_ACCOUNTING": bytes_processed == int(language_bytes.sum()) and bytes_processed > 0 and expected_tokens == tokens_processed,\n        "025_NATIVE_TRAINING": config["local_batch_per_chip"] == 2 and config["sequence_length"] == 1024,\n        "026_COMBINED_EVALUATOR": validation["split_id"].shape[0] == validation["tokens"].shape[0],\n        "027_CPU_ACCOUNTING": "byte_weights" not in training and "domain_ids" not in training and "language_ids" not in training,\n        "028_WEIGHTS_ONLY": saved["metadata"]["optimizer_resume_supported"] is False,\n    }\n    checks = {prefix_id + "-" + key: value for key, value in checks.items()}\n    checks.update({key: value == "PASS" for key, value in read_json(output / "preparation_requirements.json").items()})\n    downloads = read_json(output / "parallel_downloads.json")\n    elapsed = time.monotonic() - config["notebook_start_monotonic"]\n    rate = config["compute_units_per_hour"]\n    baseline_applicable = (identity["tokenizer_sha256"] == "7d4046bf0505a327dd5a0abbb427ecd4fc82f99c2ceaa170bc61ecde12809b0c"\n        and sha256_file(data_dir / "manifest.json") == "74d6d6305b893aa9403e8ac30c10b9d920c23bc7d3c9d020debbd5b3500d707a")\n    goals = {"tokens_per_second_at_least_45559": tokens_processed / training_seconds >= 45559.448,\n        "compilation_seconds_at_most_60": compilation_seconds <= 60.,\n        "notebook_seconds_at_most_300": elapsed <= 300.,\n        "occupancy_at_least_95_percent": tokens_processed / (updates * 2 * 1024) >= .95,\n        "finewiki_macro_bpb_at_most_v34": after["wiki"]["language_macro_bpb"] <= 2.782397528489431\n            if baseline_applicable else None}\n    improvement = ("PASS" if all(value is True for value in goals.values()) else "FAIL") if baseline_applicable else "NOT_COMPARABLE"\n    result = {"version": version, "protocol": PROTOCOL, "run_count": 1, "config": config,\n        "device": environment["device_kind"], "model_parameters": parameter_count, "optimizer": partition,\n        "source_downloads": {key: downloads[key] for key in\n            ("source_jobs", "max_workers", "peak_active_source_jobs", "fetch_seconds", "raw_cache_reused")},\n        "initial_backbone_sha256": backbone_digest, "input_transfer_seconds": input_transfer_seconds,\n        "resident_training_bytes": input_bytes, "initialization_seconds": initialization_seconds,\n        "kernel_verification_seconds": protocol_report["seconds"], "compilation_seconds": compilation_seconds,\n        "compilation_timing_scope": "one training and one combined evaluation executable plus zero-update JIT dispatch",\n        "dispatch_preparation": read_json(output / "dispatch_preparation.json"),\n        "initial_eval_seconds": initial_eval_seconds, "final_eval_seconds": final_eval_seconds,\n        "training_seconds": training_seconds,\n        "training_overrun_seconds": max(0., training_seconds - config["training_seconds"]),\n        "last_dispatch_start_seconds": last_start, "host_dispatches": len(log_rows),\n        "updates": updates, "scored_tokens": tokens_processed,\n        "tokens_per_second": tokens_processed / training_seconds,\n        "processed_original_bytes_attributed": bytes_processed,\n        "original_bytes_per_second_attributed": bytes_processed / training_seconds,\n        "byte_accounting_note": "CPU batch-prefix sums using tokenizer offset attribution; repeated passes included, not unique bytes",\n        "training_domain_byte_shares": {name: int(domain_bytes[index]) / bytes_processed for index, name in enumerate(DOMAINS)},\n        "training_language_byte_shares": {name: int(language_bytes[index]) / bytes_processed for index, name in enumerate(LANGUAGES)},\n        "training_occupancy": tokens_processed / (updates * 2 * 1024),\n        "prepared_occupancy": tokenization["splits"]["train"]["occupancy"],\n        "completed_data_passes": updates // training["tokens"].shape[0],\n        "training_batches": training["tokens"].shape[0], "data_order_sha256": orders,\n        "data_manifest_sha256": sha256_file(data_dir / "manifest.json"),\n        "initial_eval": before, "final_eval": after,\n        "primary_metric": "FineWiki mean of 12 per-language BPB values; lower is better",\n        "checkpoint_kind": "weights_only", "optimizer_resume_supported": False,\n        "checkpoint_save_verify_seconds": checkpoint_seconds, "checkpoint_bytes": checkpoint.stat().st_size,\n        "checkpoint_sha256": digest, "checkpoint_array_leaves": leaves,\n        "memory_stats": memory_stats, "compiled_memory_estimate": compiled_memory,\n        "memory_scope": "Allocator snapshot after evaluation; compiler estimate separately",\n        "notebook_elapsed_seconds": elapsed,\n        "compute_units_estimate": {"rate_per_hour": rate, "training": training_seconds * rate / 3600,\n            "through_report": elapsed * rate / 3600, "overhead_through_report": (elapsed - training_seconds) * rate / 3600},\n        "cost_note": "2.92 CU/hour supplied by user; connected runtime after report adds cost; no USD conversion assumed",\n        "historical_v2_tokens_per_second": 45559.448,\n        "historical_comparison_note": "V2 used different data and vocabulary; V3.4 used different kernels/batch/optimizer; neither isolates an optimizer effect",\n        "v34_baseline_applicable": baseline_applicable, "performance_goals": goals,\n        "performance_goal_status": improvement,\n        "requirements": {key: "PASS" if passed else "FAIL" for key, passed in checks.items()},\n        "engineering_status": "PASS" if all(checks.values()) else "FAIL",\n        "learning_status": "PASS" if after["wiki"]["language_macro_bpb"] < before["wiki"]["language_macro_bpb"] else "FAIL"}\n    write_json(output / "result.json", result)\n    write_json(output / "attempt_status.json", {"version": version, "stage": "result saved",\n        "timed_training_started": True, "optimizer_updates": updates})\n    print(json.dumps({name: result[name] for name in ("engineering_status", "learning_status",\n        "performance_goal_status", "updates", "training_seconds", "tokens_per_second",\n        "original_bytes_per_second_attributed")}, indent=2), flush=True)\n    print(f"FineWiki macro BPB: {before[\'wiki\'][\'language_macro_bpb\']:.6f} -> {after[\'wiki\'][\'language_macro_bpb\']:.6f}", flush=True)\n    if not all(checks.values()):\n        raise RuntimeError(version + " engineering requirement failed; result.json was saved.")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--phase", choices=["preflight", "prepare", "train"], required=True)\n    parser.add_argument("--work", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    try:\n        if args.phase == "preflight":\n            info = preflight()\n            write_json(args.output / "preflight.json", info)\n            print(json.dumps({name: info[name] for name in ("platform", "device_kind", "device_count", "jax")}, indent=2))\n        elif args.phase == "prepare":\n            prepare_cached(read_json(args.work / "config.json"), args.work, args.output)\n        else:\n            cache = args.work.parent / "jax-cache-V3"\n            cache.mkdir(parents=True, exist_ok=True)\n            os.environ["JAX_COMPILATION_CACHE_DIR"] = str(cache)\n            os.environ["JAX_PERSISTENT_CACHE_MIN_COMPILE_TIME_SECS"] = "1"\n            try:\n                train_pilot(args.work, args.output)\n            finally:\n                persistent = args.output.parent.parent / "V3-cache" / "jax"\n                try:\n                    persistent.mkdir(parents=True, exist_ok=True)\n                    shutil.copytree(cache, persistent, dirs_exist_ok=True)\n                except OSError as error:\n                    write_json(args.output / "cache_warning.json", {"warning": str(error), "training_retried": False})\n    except Exception:\n        if (args.output / "failure.json").exists():\n            write_json(args.output / ("failure-secondary-" + str(time.time_ns()) + ".json"),\n                {"phase": args.phase, "error": traceback.format_exc(), "automatic_retry": False})\n            raise\n        write_json(args.output / "failure.json", {"phase": args.phase, "error": traceback.format_exc(), "automatic_retry": False})\n        raise\n\n\nif __name__ == "__main__":\n    main()\n'
CONFIG = {'experiment': 'GEM-COLAB-V3.6', 'version': 'V3.6', 'implementation_revision': 'V3.6-native-1', 'requirement_prefix': 'V36', 'model_id': 'google/gemma-3-270m', 'tokenizer_reference': 'google/gemma-3-270m', 'tokenizer_revision': '9b0cfec892e2bc2afd938c98eabe4e4a7b1e0ca1', 'sequence_length': 1024, 'local_batch_per_chip': 2, 'updates_per_dispatch': 16, 'training_seconds': 60.0, 'minimum_updates': 5, 'seed': 42, 'expected_devices': 1, 'optimizer_label': 'Muon', 'learning_rate': 0.0015, 'warmup_updates': 5, 'gradient_clip': 1.0, 'weight_decay': 0.01, 'compute_units_per_hour': 2.92, 'checkpoint_kind': 'weights_only', 'muon_beta': 0.95, 'muon_ns_steps': 5, 'muon_consistent_rms': 0.2}
CONFIG["tokenizer_reference"] = TOKENIZER_REFERENCE.strip()
CONFIG["tokenizer_revision"] = "9b0cfec892e2bc2afd938c98eabe4e4a7b1e0ca1" if CONFIG["tokenizer_reference"] == "google/gemma-3-270m" else "main"
CONFIG["notebook_start_monotonic"] = NOTEBOOK_START_MONOTONIC
CONFIG["setup_seconds"] = SETUP_SECONDS
PROGRAM_PATH = WORK / "pilot_V3_6.py"
PROGRAM_PATH.write_text(PROGRAM_SOURCE)
(WORK / "config.json").write_text(json.dumps(CONFIG, indent=2))
shutil.copyfile(PROGRAM_PATH, OUTPUT / "pilot_source.py")
shutil.copyfile(WORK / "config.json", OUTPUT / "requested_config.json")

def run_phase(phase, token=None):
    if phase == "train" and (OUTPUT / "training.started").exists():
        if (OUTPUT / "result.json").exists():
            print("Existing completed attempt; displaying result without retraining.")
            print((OUTPUT / "result.json").read_text())
            return
        raise RuntimeError("This folder already owns an attempt. Prior logs preserved. Inspect attempt_status.json and train.log; do not delete the marker blindly.")
    if phase == "prepare" and (WORK / "prepared" / "resolved_config.json").exists():
        print("Preparation already completed for this run; reusing it.")
        return
    environment = dict(os.environ)
    environment.pop("PYTHONPATH", None)
    environment.pop("HF_TOKEN", None)
    if token:
        environment["HF_TOKEN"] = token
    environment["TOKENIZERS_PARALLELISM"] = "false"
    environment["JAX_TRACEBACK_FILTERING"] = "off"
    environment["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"
    environment["HF_HUB_ETAG_TIMEOUT"] = "30"
    environment["PYTHONUNBUFFERED"] = "1"
    environment["MPLCONFIGDIR"] = str(BASE / "matplotlib-cache")
    command = [str(PYTHON), str(PROGRAM_PATH), "--phase", phase, "--work", str(WORK), "--output", str(OUTPUT)]
    log_path = OUTPUT / (phase + ".log")
    if log_path.exists():
        raise RuntimeError(f"Preserving existing {phase} log; this phase is not retried automatically.")
    with log_path.open("x") as log:
        process = subprocess.Popen(command, env=environment, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        status = process.wait()
    if status:
        raise RuntimeError(f"V3.6 {phase} failed. See {log_path} and failure.json. No automatic retry occurred.")


## 4. Reuse the frozen corpus and tokenize
Verified existing raw/token caches are reused. Cold preparation retains six source workers and thirty source jobs.


In [ ]:
PREPARATION_START = time.monotonic()
run_phase("prepare", token=HF_TOKEN)
HF_TOKEN = None
PREPARATION_SECONDS = time.monotonic() - PREPARATION_START
print((OUTPUT / "preparation.json").read_text())
print((OUTPUT / "parallel_downloads.json").read_text())


## 5. Verify the TPU, compile, evaluate, then train once
Hardware verification occurs once in this process. Compilation, initial evaluation and zero-update JIT preparation precede timing. Final evaluation and FP32 weights saving follow the timed interval. No automatic retry.


In [ ]:
run_phase("train")


## 6. Read measured results
Engineering, learning and performance-target results are separate. Targets are not promises.


In [ ]:
from IPython.display import display, Markdown
result = json.loads((OUTPUT / "result.json").read_text())
rows = [
    ("Version", result["version"]), ("Optimizer", result["optimizer"]["name"]),
    ("Muon / AdamW parameters", f'{result["optimizer"]["muon_parameters"]:,} / {result["optimizer"]["adamw_parameters"]:,}'),
    ("Engineering / learning / target status", f'{result["engineering_status"]} / {result["learning_status"]} / {result["performance_goal_status"]}'),
    ("Training attempts", result["run_count"]), ("Model parameters", f'{result["model_parameters"]:,}'),
    ("Actual training seconds", f'{result["training_seconds"]:.3f}'),
    ("Overrun seconds", f'{result["training_overrun_seconds"]:.3f}'),
    ("Updates", f'{result["updates"]:,}'), ("Useful tokens/s", f'{result["tokens_per_second"]:,.1f}'),
    ("Attributed original bytes/s", f'{result["original_bytes_per_second_attributed"]:,.1f}'),
    ("Useful occupancy", f'{result["training_occupancy"]:.2%}'),
    ("Initial FineWiki macro BPB", f'{result["initial_eval"]["wiki"]["language_macro_bpb"]:.6f}'),
    ("Final FineWiki macro BPB", f'{result["final_eval"]["wiki"]["language_macro_bpb"]:.6f}'),
    ("Initial / final in-mixture BPB", f'{result["initial_eval"]["in_mix"]["pooled_bpb"]:.6f} / {result["final_eval"]["in_mix"]["pooled_bpb"]:.6f}'),
    ("Setup / preparation seconds", f'{SETUP_SECONDS:.1f} / {PREPARATION_SECONDS:.1f}'),
    ("Compilation seconds", f'{result["compilation_seconds"]:.1f}'),
    ("Initial / final evaluation seconds", f'{result["initial_eval_seconds"]:.1f} / {result["final_eval_seconds"]:.1f}'),
    ("Weights save/verification seconds", f'{result["checkpoint_save_verify_seconds"]:.1f}'),
    ("Weights artifact GB", f'{result["checkpoint_bytes"] / 1e9:.3f}'),
    ("Exact optimizer resume", "Not included: weights-only artifact"),
    ("Notebook seconds through report", f'{result["notebook_elapsed_seconds"]:.1f}'),
    ("Estimated training CU", f'{result["compute_units_estimate"]["training"]:.5f}'),
    ("Estimated CU through report", f'{result["compute_units_estimate"]["through_report"]:.5f}'),
]
display(Markdown("| Measurement | Result |\n|---|---|\n" + "\n".join(f"| {name} | {value} |" for name, value in rows)))
language_rows = []
for language, final in result["final_eval"]["wiki"]["languages"].items():
    initial = result["initial_eval"]["wiki"]["languages"][language]["bpb"]
    language_rows.append(f"| {language} | {initial:.6f} | {final['bpb']:.6f} | {final['bpb'] - initial:+.6f} |")
display(Markdown("| FineWiki language | Initial BPB | Final BPB | Change |\n|---|---:|---:|---:|\n" + "\n".join(language_rows)))
print("Timed source byte shares:", json.dumps(result["training_domain_byte_shares"], indent=2))
print("Timed language byte shares:", json.dumps(result["training_language_byte_shares"], indent=2))
print("Requirements:", json.dumps(result["requirements"], indent=2))
print("Performance targets (not promises):", json.dumps(result["performance_goals"], indent=2))
print("Artifacts:", OUTPUT)
print("2.92 CU/hour is an estimate. Disconnecting remains necessary to stop further connected-runtime use.")


## 7. Download reports and disconnect
The report ZIP excludes the large weights artifact, which remains in Drive with the tokenizer.


In [ ]:
import zipfile
from google.colab import files
REPORT_ZIP = WORK / "gemma_colab_pilot_V3_6_reports.zip"
with zipfile.ZipFile(REPORT_ZIP, "w", zipfile.ZIP_DEFLATED) as bundle:
    for name in ["result.json", "environment.json", "preflight.json", "requested_config.json", "config.json",
        "data_manifest.json", "tokenization.json", "tokenizer_identity.json", "source-lock.json",
        "preparation.json", "parallel_downloads.json", "preparation_requirements.json", "dispatch_preparation.json",
        "attempt_status.json", "timed_training.started.json", "dependency_source.json", "backbone_initialization.json",
        "optimizer_partition.json", "optimizer_parameter_routing.json", "kernel_verification.json", "training_accounting.json",
        "training.jsonl", "checkpoint_manifest.json", "requirements.in", "requirements.lock", "pilot_source.py",
        "prepare.log", "train.log", "setup-install.log", "setup-bootstrap.log", "setup-python.log", "cache_warning.json", "failure.json"]:
        path = OUTPUT / name
        if path.is_file():
            bundle.write(path, name)
shutil.copyfile(REPORT_ZIP, OUTPUT / REPORT_ZIP.name)
files.download(str(REPORT_ZIP))
print("FP32 weights:", OUTPUT / "weights.npz")
print("Tokenizer files:", OUTPUT / "tokenizer")
print("Finished. Select Runtime > Disconnect and delete runtime when done.")


# V3.6 data preparation

## Fixed data mixture and scientific limits

| Training source | Original UTF-8 byte share |
|---|---:|
| FinePDFs-Edu via Smol-Data | 27.5% |
| DCLM via Smol-Data | 16.5% |
| FineWeb-Edu via Smol-Data | 11.0% |
| FineWeb2, 11 non-English languages | 30.0% |
| FineMath `finemath-3plus` | 5.0% |
| CodeParrot clean Python | 5.0% |
| FineTranslations, original text + English translation | 5.0% |
| **Total** | **100.0%** |

The first three sources use [Smol-Data's globally shuffled English mixture](https://huggingface.co/datasets/HuggingFaceFW/finepdfs_edu_50BT-dclm_30BT-fineweb_edu_20BT-shuffled). Its upstream 50/30/20 recipe uses reference-token counts; V3 reads its source labels and enforces **27.5/16.5/11.0% original-byte quotas itself**. It does not assume token shares equal byte shares.

FineWeb2's 30% is split: French **3%**, Spanish **3%**, Portuguese **2.4%**, Chinese **4.5%**, Arabic **4.5%**, Hindi **4.5%**, Swahili **2.4%**, Zulu **1.5%**, Xhosa **1.5%**, Setswana **1.5%**, Sesotho **1.2%** of total training bytes. These are declared screening priorities, including Southern African languages, not measured optimal weights.

Translations allocate **1.25%** of total bytes to each French/Arabic/Hindi/Swahili source bucket, including its English side. Sides are separate documents with one shared source-group split; neither can attend to the other. Only educational score **≥2** and non-early-stopped rows are accepted. These English sides are synthetic translations, unlike the extracted PDF source. Actual language exposure is reported rather than assumed to be 50/50.

FineWiki reserves **64 KiB per language** across English plus the 11 languages above, approximately **768 KiB total**. It is a separate domain holdout. The in-mixture set contains approximately **256 KiB**, allocated by the training source weights. UTF-8 boundaries allow up to **3 bytes shortfall per bucket**. Selected source spans are at most **32 KiB**; clipping preserves the exact source prefix, with no normalization.

Canonical URL/source groups use a deterministic hash split: **10% of groups eligible for evaluation, 90% for training**; the evaluation byte quota is much smaller than 10% of the training quota. Code groups are whole repositories. Global exact source/span hashes remove exact repeats across splits, and Wikipedia URLs are excluded from non-Wiki inputs. This does not prove absence of near duplicates or web copies without identifiable URLs.

Dataset commits are resolved **once on first preparation**, saved in the shared Drive source lock, and reused with frozen raw hashes for all tokenizer runs. Sampling uses bounded deterministic stream prefixes, followed by seeded packing/order; it is not a uniform sample of each full corpus. Realized 60-second source shares can differ from pool weights and are recorded.

The [OLMo 3 paper](https://arxiv.org/abs/2512.13961) motivates high-quality web/PDF/code/math coverage and byte-based evaluation. The [dataset-mixing article](https://huggingface.co/blog/codelion/optimal-dataset-mixing) and [Smol-Data collection](https://huggingface.co/collections/HuggingFaceFW/smol-data) inform the static English mixture. **The final multilingual weights are an engineering choice, not a reproduced optimal-mixture result.** V3 performs no mixture search, optimizer sweep or multiple pilot runs.

Primary BPB is `sum(original-text token negative log probabilities) / (original UTF-8 bytes × ln(2))`. Virtual EOS loss is reported separately. For example **69,314.7 nats over 100,000 bytes → 1 BPB**. Language macro BPB gives each of the 12 languages equal weight; pooled BPB weights them by bytes. Accumulation is FP32. All candidates score the same text, but a 1,024-token context covers different byte lengths; this is a fixed-token-context, fixed-time screen.

Vocabulary IDs are densely remapped with separate PAD/BOS/EOS protocol IDs. Total parameters change by **640 per added vocabulary row**; the backbone stays identical, and its initial parameter SHA256 must match the shared baseline. The notebook fails if a tokenizer changes selected text on decode or exceeds the 300M limit. It supports Hugging Face `tokenizer.json` artifacts without remote tokenizer code.

The timed throughput report includes tokens/s and **attributed original bytes/s**. Byte attribution uses the monotonic original-character offset frontier; each source byte is counted once per full pass. It is not a claim that every tokenizer token has an independent byte boundary. Repeated passes count as repeated processed bytes. This one-seed, one-minute result can reject obvious failures; it cannot establish a final scientific tokenizer ranking.

V2's recorded **45,559 tokens/s** used different data, batch, optimizer and vocabulary handling. V3 reports that historical number only as context. It does not claim a controlled speed ratio or compare old token loss with new BPB.

V3.6 fetches source prefixes concurrently, then merges in the same canonical source order. Its initial cache target is **125% of each byte quota**, with at least **4 candidate spans per split/source bucket**, to allow global deduplication without reopening most streams. Only the original byte quotas enter the final corpus. If the buffer is insufficient, the same pinned stream continues from its committed cursor, within the original row limit and **256 MiB per-source cached JSONL cap**. The cap bounds cached candidate bytes, not HTTP traffic or Parquet row-group read sizes.

Source batches target **4 MiB** and commit after complete input rows, so a batch can exceed that target by one row. Commit attempts also occur after **5 seconds of row processing**; a blocked network read cannot be checkpointed mid-read. An interrupted job resumes at its saved row cursor; `skip()` can reread earlier source data when the remote format cannot seek directly. Completed caches avoid refetching those jobs. The original V3 implementation did not persist individual source samples before finalizing the corpus, so its unfinished sequential progress cannot be recovered from console logs. A complete, verified V3 raw corpus remains reusable.

